# The CW sine lineage — replacing tanh, on the road to σ-binary

The longest attack investigation in this repository, and the direct ancestor of the thesis method. The problem: a Carlini–Wagner-style attack optimises a continuous perturbation, but malware features are **binary** — the perturbation must be squashed into [0, 1] during optimisation and rounded at the end, without killing the gradients. The classic squashing function is tanh; this lineage replaces it with a **sine-based function** and then spends most of its effort on the two questions that decide whether the attack works: the **learning rate**, and **how and when to round**.

The notebook follows the lineage in order: the idea and the first runs; the evaluation harness (rounded vs non-rounded scoring, the fixed-features check); the systematic study (learning-rate range test, rounding at the end vs in the loop, one constraint vs all constraints, L2/L1/tanh distances); and the final form, with its long series of loss-function variants. Repeated runs of the same configuration are shown once.

This is where the thesis's σ-binary method comes from: `06-thesis-evaluation/` measures the mature version, and the final published method lives in the separate sigma-binary repository.

*Consolidated from `CW_sine.ipynb`, `CW_sine_autoLR.ipynb`, `CW_sine_complete_old.ipynb` and `CW_sine5.ipynb` (originally in `CW-CL`, `Adversarials` and `a_malware2`). Outputs cleared.*

## Step 1 — Setup (shared by the whole lineage)

# Load dataset and libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import time
import os
import math
from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd
import tqdm
import pickle

torch.manual_seed(0)
np.random.seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/best_model_AT_imax.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/data/model_AT_max%20(4).pth',
]

output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
from adverserial_attacks_functions import *

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array, dtype=torch.uint8).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array, dtype=torch.uint8).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 128
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Create an instance of your model
model_AT_rFGSM = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM.load_state_dict(torch.load('model_AT_rFGSM.pth', map_location=torch.device(device)))

# Create an instance of your model
model_AT_imax = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_imax.load_state_dict(torch.load('best_model_AT_imax.pth', map_location=torch.device(device)))

In [ ]:
# Define the Linear Classifier(NO NEED)
class LinearClassifier(nn.Module):
    def __init__(self, dim_in=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(dim_in, num_classes)

    def forward(self, x):
        return self.fc(x)


# Define the CoreModel
class CoreModel(nn.Module):
    def __init__(self):
        super(CoreModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.dropout1 = nn.Dropout(0.6)
        self.fc2 = nn.Linear(200, 200)
        self.dropout2 = nn.Dropout(0.6)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x, return_z=False):
        x = F.relu(self.fc1(x))
        #x = self.dropout1(x)
        z = F.relu(self.fc2(x))
        z2 = self.dropout2(z)
        output  = self.fc3(z2)
        if return_z:
            return output, z

        return output

    def forward_from_z(self, z):
        output = self.fc3(z)
        return output

# Define the Projection Head
class ProjHead(nn.Module):
    def __init__(self, dim_in=200, feat_dim=128):
        super(ProjHead, self).__init__()
        self.head = nn.Linear(dim_in, feat_dim)

    def forward(self, x):
        #x = F.normalize(x, dim=1)
        z_proj = self.head(x)
        z_proj = F.normalize(z_proj, dim=1)
        return z_proj



# Define the Wrapper
class Wrapper(nn.Module):
    def __init__(self, CoreModel, ProjLayer):
        super(Wrapper, self).__init__()
        self.CoreModel = CoreModel
        self.ProjLayer = ProjLayer
        #self.LC = LC

    def forward(self, x, return_z=False):
        output, z = self.CoreModel(x, return_z=True)
        z_proj = self.ProjLayer(z)
        #output = self.LC(z)
        if return_z:
            return output, z_proj
        else:
            return output

# Instantiate the components
core_model = CoreModel()
proj_head = ProjHead(200, 128)
#linear_classifier = LinearClassifier(128, 2)

# Combine them in the Wrapper
combined_model = Wrapper(core_model, proj_head)

# Example usage
x = torch.randn(2, 10000)  # Example input
output, z_proj = combined_model(x,return_z=True)
print(output.shape)
print(z_proj.shape)

In [ ]:
def pgd_min22(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        #loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        #loss = torch.clamp((real - other + 0.1), min=0.0)
        loss = (real - other)


        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    #x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    #loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    #replace_flag = (loss_adv > loss_natural).squeeze()
    #x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

## Part 1 — The idea: a sine function instead of tanh

### Step 2 — First attempts: arctanh, and why it struggles

#CW

In [ ]:
import torch
# Define the arctanh functions with numerical stability
def torch_arctanh1(x, eps=1e-5):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

In [ ]:
a = torch.tensor(0.)
torch_arctanh1(a)

In [ ]:
(torch.tanh(torch_arctanh1(a))/0.99999) * 0.5 + 0.5

In [ ]:
b = torch.tensor(0.)
torch_arctanh0(b)

In [ ]:
(torch.tanh(torch_arctanh0(b))/1.) * 0.5 + 0.5

### Step 3 — The first CW run with the new machinery

In [ ]:
import torch
import torch.optim as optim
import numpy as np
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

target_label = torch.zeros_like(mal_y_batch.squeeze(), dtype=torch.int64)

def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh1(x, eps=1e-2):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

model = model_AT_rFGSM.to(device)
imgs = mal_x_batch
labs = target_label
targeted = True
#abort_early = True
confidence = 0.01
initial_const = 0.1
learning_rate = 0.1
binary_search_steps = 1
max_iterations = 1000
removal_array = removal_array

batch_size = imgs.shape[0]

expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

# Calculate indices where mask is 0 or 1
mask_0 = non_fixed_features_mask == 0
mask_1 = non_fixed_features_mask == 1

# Create a tensor with values based on the boolean condition
factor_tensor = torch.where(mask_1, torch.tensor(0.99), torch.tensor(1.0))

# Create empty tensor for the result
imgs_tanh = torch.empty_like(imgs)

# Apply the transformations based on the mask
imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
imgs_tanh[mask_1] = torch_arctanh1(imgs[mask_1])

# Use clone().detach() to copy the tensor correctly
imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# Set the lower and upper bounds for binary search
lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

# Variables to store the best results
o_bestl2 = torch.full((batch_size,), 1e10, device=device)
o_bestscore = torch.full((batch_size,), -1, device=device)
o_bestattack = imgs.clone().detach()
# o_best_loss1: for getting best loss1 for those samples with no successful attack
o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
o_best_unsuccessfull_attack = imgs.clone().detach()

# Set model to evaluation mode
model.eval()
#criterion = nn.CrossEntropyLoss(reduction='none')

for outer_step in range(binary_search_steps):

    if outer_step == (binary_search_steps - 1):
        max_iterations = 3000
        #learning_rate = 0.005

    print(f"Binary search step {outer_step}, max_iterations: {max_iterations}")

    # Initialize modifier with requires_grad=True
    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

    # Setup optimizer
    optimizer = optim.Adam([modifier], lr=learning_rate)

    # Setup learning rate scheduler
    #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

    # To track the best loss for each sample and count of iterations without improvement
    best_loss = torch.full((batch_size,), float('inf'), device=device)
    no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

    # Track active samples
    active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

    bestl2 = torch.full((batch_size,), 1e10, device=device)
    bestscore = torch.full((batch_size,), -1, device=device)

    for iteration in range(max_iterations):
        if active_mask.sum() == 0:
            print(f"All samples are inactive at iteration {iteration}.")
            break

        # Filter active samples
        active_indices = torch.where(active_mask)[0]
        active_modifier = modifier[active_mask]
        active_imgs_tensor = imgs_tensor[active_mask]
        active_labs_tensor = labs_tensor[active_mask]
        active_CONST = CONST[active_mask]

        optimizer.zero_grad()

        # Compute new images in original space
        modified_input = active_imgs_tensor + active_modifier
        newimg = (torch.tanh(modified_input) / factor_tensor[active_mask]) * 0.5 + 0.5

        break

In [ ]:
def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh((2 * x - 1) * (1 - eps))

def torch_arctanh1(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh((2 * x - 1) * (1 - eps))

def CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, cons_increase_factor=2, device=device):

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]

    # Ensure fixed_features_mask is a tensor
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    # Calculate indices where mask is 0 or 1
    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    # Create a tensor for correct the value based on the element which is converted from torch_arctanh0 or torch_arctanh1
    corrected_factor = torch.where(mask_1, torch.tensor(0.99999999), torch.tensor(1.0))


    # Create empty tensor for the result
    imgs_tanh = torch.empty_like(imgs)

    # Apply the transformations based on the mask
    imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
    imgs_tanh[mask_1] = torch_arctanh1(imgs[mask_1])

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Set the lower and upper bounds for binary search
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Variables to store the best results
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    # o_best_loss1: for getting best loss1 for those samples with no successful attack
    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()

    # Set model to evaluation mode
    model.eval()

    for outer_step in range(binary_search_steps):

        '''if outer_step == (binary_search_steps - 1):
            max_iterations = 3000
            learning_rate = 0.01'''

        # Initialize modifier with requires_grad=True
        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

        # Setup optimizer
        optimizer = optim.Adam([modifier], lr=learning_rate)

        # Setup learning rate scheduler
        #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

        # Track active samples
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if active_mask.sum() == 0:
                print(f"All samples are inactive at iteration {iteration}.")
                break

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier
            newimg = (torch.tanh(modified_input) ) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            #l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (active_CONST * loss1) + l2dist
            total_loss = torch.sum(loss1)

            # Backpropagate the loss
            total_loss.backward()

            optimizer.step()

            with torch.no_grad():
                # Check for improvement in loss for each sample
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

                # Check for improvement in loss1 for each sample
                improved_mask2 = (loss1 < o_best_loss1[active_mask])&(l2dist < 10.)
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                # Update the best result found so far
                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1) :
          # Create masks for successful and unsuccessful attacks
          success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
          print('success_mask ', success_mask)
          failure_mask = ~success_mask

          # Update upper bounds and constants where attacks are successful
          upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

          # Update lower bounds and constants where attacks are unsuccessful
          lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

          # Handle cases where upper bound is still too high
          upper_limit_mask = upper_bound < 1e9
          CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / cons_increase_factor
          CONST[failure_mask & ~upper_limit_mask] *= 2

    # Final output based on the mask condition
    # our goal is to perturb all samples even those without successful attack
    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            print(outputs)
            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            print((y_pred == 0))
            print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            print('***************************')
            break


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return pertb_mal_x

In [ ]:
#CW(mal_x_batch, mal_y_batch, model_AT_rFGSM, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, device=device)
attack_params =  {'device':device, 'confidence':1., 'removal_array':removal_array,'max_iterations': 3000, 'learning_rate': 0.01, 'initial_const': 100 , 'binary_search_steps' : 5, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, combined_model, CW, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 1000, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model, pgd_min22, device, **attack_params)

### Step 4 — Designing the sine function and its derivative

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import newton

# Define the function
def f(x):
    return 3 * x**2 - 2 * x**3

# Generate values for x from 0 to 1
x_values = np.linspace(-0.5, 2, 500)
y_values = f(x_values)

# Plot the function
plt.figure(figsize=(10, 6))
plt.plot(x_values, y_values, label=r"$f(x) = 3x^2 - 2x^3$", color="blue")

# Add labels and title
plt.title("Plot of the Function $f(x) = 3x^2 - 2x^3$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Define the function and its gradient
def sin_func(x):
    return (np.sin(x-(np.pi/2.))+1)/2.

def sin_func_prime(x):
    return np.cos(x-(np.pi/2.))/2.

# Generate values for x from -2π to 2π
x_values = np.linspace(-2 * np.pi, 2 * np.pi, 1000)
sin_values = sin_func(x_values)
sin_prime_values = sin_func_prime(x_values)

# Plot sin(x)
plt.figure(figsize=(10, 6))
plt.plot(x_values, sin_values, label=r"$\sin(x)$", color="blue")

# Add labels and title
plt.title("Plot of $\sin(x)$")
plt.xlabel("x")
plt.ylabel("$\sin(x)$")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.grid(True)
plt.legend()
plt.show()

# Plot the gradient of sin(x)
plt.figure(figsize=(10, 6))
plt.plot(x_values, sin_prime_values, label=r"$\cos(x)$", color="red")

# Add labels and title
plt.title("Plot of the Gradient of $\sin(x)$: $\cos(x)$")
plt.xlabel("x")
plt.ylabel("$\cos(x)$")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
eps = 1e-4
def sin_func(x):
    return ((np.sin((x-0.5)*(np.pi))* (1 - eps))+1)/2.

In [ ]:
sin_func(0)

In [ ]:
sin_func(1)

In [ ]:
def sin_func_prime(x):
    return np.pi/2. * np.cos((x-0.5)*(np.pi))* (1 - eps)

In [ ]:
sin_func_prime(0.99995)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Small epsilon value
eps = 1e-4

# Define the function with the perturbation
def sin_func(x):
    return ((np.sin((x-0.5)*(np.pi)) * (1 - eps)) + 1) / 2.

# Define the gradient of the function using numerical differentiation
def sin_func_gradient(x, eps=1e-5):
    return np.pi/2. * np.cos((x-0.5)*(np.pi))* (1 - eps)

# Generate x values between -0.5 and 1.5
x_values = np.linspace(-0.5, 1.5, 1000)
sin_values = sin_func(x_values)
gradient_values = sin_func_gradient(x_values)

# Plot the function
plt.figure(figsize=(10, 6))
plt.plot(x_values, sin_values, label=r"$\sin\left((x-0.5)\pi\right)$ function", color="blue")
plt.title("Plot of the function")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.grid(True)
plt.legend()
plt.show()

# Plot the gradient of the function
plt.figure(figsize=(10, 6))
plt.plot(x_values, gradient_values, label="Gradient of the function", color="red")
plt.title("Plot of the gradient of the function")
plt.xlabel("x")
plt.ylabel("Gradient f'(x)")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

In [ ]:
sin_func_torch(a, eps=1e-4)

### Step 5 — Runs, and rounding the result (final_solution)

In [ ]:
import torch
import torch.optim as optim
import numpy as np
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

target_label = torch.zeros_like(mal_y_batch.squeeze(), dtype=torch.int64)

def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

# Define the function in PyTorch
#def sin_func_torch(x, eps=1e-4):
    #return ((torch.sin((x - 0.5) * torch.pi) * (1 - eps)) + 1) / 2.0

def sin_func_torch(input_tensor):
    output = torch.where(torch.isinf(input_tensor), torch.zeros_like(input_tensor), input_tensor)
    output = (torch.sin((output - 0.5) * torch.pi) + 1) / 2.0
    return output


model = model_AT_rFGSM.to(device)
imgs = mal_x_batch
labs = target_label
targeted = True
#abort_early = True
confidence = 0.01
initial_const = 0.1
learning_rate = 0.1
binary_search_steps = 4
max_iterations = 1000
removal_array = removal_array

batch_size = imgs.shape[0]

expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

# Calculate indices where mask is 0 or 1
mask_0 = non_fixed_features_mask == 0
mask_1 = non_fixed_features_mask == 1

# Create a tensor for correct the value based on the element which is converted from torch_arctanh0 or torch_arctanh1
corrected_factor = torch.where(mask_1, torch.tensor(0.99), torch.tensor(1.0))

# Create empty tensor for the result
#imgs_tanh = imgs.clone().detach().to(dtype=torch.float32, device=device)
imgs_tanh = torch.empty_like(imgs)

# Apply the transformations based on the mask
imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0], eps=1e-12)
imgs_tanh[mask_1] = imgs[mask_1]

# Use clone().detach() to copy the tensor correctly
imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# Set the lower and upper bounds for binary search
lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

# Variables to store the best results
o_bestl2 = torch.full((batch_size,), 1e10, device=device)
o_bestscore = torch.full((batch_size,), -1, device=device)
o_bestattack = imgs.clone().detach()
# o_best_loss1: for getting best loss1 for those samples with no successful attack
o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
o_best_unsuccessfull_attack = imgs.clone().detach()

# Set model to evaluation mode
model.eval()
#criterion = nn.CrossEntropyLoss(reduction='none')
gradients = []
for outer_step in range(binary_search_steps):

    if outer_step == (binary_search_steps - 1):
        max_iterations = 3000
        #learning_rate = 0.005

    print(f"Binary search step {outer_step}, max_iterations: {max_iterations}")

    # Initialize modifier with requires_grad=True
    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

    # Setup optimizer
    optimizer = optim.Adam([modifier], lr=learning_rate)

    # Setup learning rate scheduler
    #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

    # To track the best loss for each sample and count of iterations without improvement
    best_loss = torch.full((batch_size,), float('inf'), device=device)
    no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

    # Track active samples
    active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

    bestl2 = torch.full((batch_size,), 1e10, device=device)
    bestscore = torch.full((batch_size,), -1, device=device)

    for iteration in range(max_iterations):
        if active_mask.sum() == 0:
            print(f"All samples are inactive at iteration {iteration}.")
            break

        # Filter active samples
        active_indices = torch.where(active_mask)[0]
        active_modifier = modifier[active_mask]
        active_imgs_tensor = imgs_tensor[active_mask]
        active_labs_tensor = labs_tensor[active_mask]
        active_CONST = CONST[active_mask]

        optimizer.zero_grad()

        # Compute new images in original space
        modified_input = active_imgs_tensor + active_modifier

        # Compute outputs of the two functions
        arctanh_output = torch.tanh(modified_input) * 0.5 + 0.5
        sin_output = sin_func_torch(modified_input)

        # Create the newimg tensor based on the condition in mask_1
        newimg = torch.where(mask_0[active_mask], arctanh_output, sin_output)

        # Get model predictions (pre-softmax)
        output = model(newimg.to(device))

        # Compute L2 distance to original images
        #l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
        l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


        # Calculate the loss based on predictions
        real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
        other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

        # Calculate adversarial loss
        loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
        #loss1 = criterion(output, active_labs_tensor)
        loss2 = l2dist
        loss = (active_CONST * loss1) + loss2
        total_loss = torch.sum(loss)

        # Backpropagate the loss
        total_loss.backward()

        #gradients.append(modifier.grad.clone().detach().cpu())


        optimizer.step()

        # Step the learning rate scheduler
        #scheduler.step()

        with torch.no_grad():

            # Check for improvement in loss for each sample
            improved_mask = loss < best_loss[active_mask]
            best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
            no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

            # Check for improvement in loss1 for each sample
            improved_mask2 = (loss1 < o_best_loss1[active_mask])&(l2dist < 10.)
            o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
            o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

            # Update the best result found so far
            update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
            bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
            bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

            update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
            o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
            o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
            o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

            # Skip the rest of the loop for samples that haven't improved in 50 iterations
            active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100
            #active_mask[active_mask.clone()] = l2dist < 5.


            # Print losses every 10% of max_iterations
            #if iteration % (max_iterations // 20) == 0:
            '''if (iteration+1) % 200 == 0:
                #print('masks ',mask_one.sum(1),mask_zero.sum(1))
                print(output)
                #print('active_mask ', active_mask.sum())
                print(f"Iteration {iteration+1}, Total Loss: {total_loss.item()}")'''

    print(f"Binary search step {outer_step}, CONST: {CONST}")
    print(f"Binary search step {outer_step}, o_bestl2: {o_bestl2}")
    print(f"Binary search step {outer_step}, output: {model(o_bestattack)}")

    if outer_step != (binary_search_steps - 1) :
        # Create masks for successful and unsuccessful attacks
        success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
        failure_mask = ~success_mask

        # Update upper bounds and constants where attacks are successful
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

        # Update lower bounds and constants where attacks are unsuccessful
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        # Handle cases where upper bound is still too high
        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= 2



# Final output based on the mask condition
# our goal is to perturb all samples even those without successful attack
success_attack_mask = (o_bestl2 != 1e10)
final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler

def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh((2 * x - 1) * (1 - eps))

def sin_func_torch(input_tensor):
    output = torch.where(torch.isinf(input_tensor), torch.zeros_like(input_tensor), input_tensor)
    output = (torch.sin((output - 0.5) * torch.pi) + 1) / 2.0
    return output

def CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, cons_increase_factor=2, device=device):

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Ensure fixed_features_mask is a tensor
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    # Calculate indices where mask is 0 or 1
    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    # Create a tensor for correct the value based on the element which is converted from torch_arctanh0 or torch_arctanh1
    #corrected_factor = torch.where(mask_1, torch.tensor(0.99999999), torch.tensor(1.0))


    # Create empty tensor for the result
    imgs_tanh = torch.empty_like(imgs)

    # Apply the transformations based on the mask
    imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
    imgs_tanh[mask_1] = imgs[mask_1]

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Set the lower and upper bounds for binary search
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Variables to store the best results
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    # o_best_loss1: for getting best loss1 for those samples with no successful attack
    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()

    # Set model to evaluation mode
    model.eval()

    for outer_step in range(binary_search_steps):

        print(CONST)

        '''if outer_step == (binary_search_steps - 1):
            max_iterations = 3000
            learning_rate = 0.01'''

        # Initialize modifier with requires_grad=True
        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

        # Setup optimizer
        #optimizer = optim.Adam([modifier], lr=learning_rate,betas=(0.1,0.99))
        optimizer = optim.Adam([modifier], lr=learning_rate)
        #optimizer = optim.SGD([modifier], lr=learning_rate)

        # Setup learning rate scheduler
        #scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=0, verbose=True, min_lr=1e-6)

        # Setup learning rate scheduler
        #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

        # Track active samples
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):

            if active_mask.sum() == 0:
                print(f"All samples are inactive at iteration {iteration}.")
                if optimizer.param_groups[0]['lr'] > 1e-4 :
                    optimizer.param_groups[0]['lr'] *= 0.5
                    print(optimizer.param_groups[0]['lr'])
                    active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)
                    best_loss = torch.full((batch_size,), float('inf'), device=device)
                else :
                    break

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier

            # Compute outputs of the two functions
            arctanh_output = torch.tanh(modified_input) * 0.5 + 0.5
            sin_output = sin_func_torch(modified_input)

            # Create the newimg tensor based on the condition in mask_1
            newimg = torch.where(mask_0[active_mask], arctanh_output, sin_output)


            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            #l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss2 = l2dist
            loss = (active_CONST * loss1) + loss2
            #loss = (loss2 / active_CONST) + loss1
            total_loss = torch.sum(loss)

            # Backpropagate the loss
            total_loss.backward()

            optimizer.step()

            with torch.no_grad():
                # Check for improvement in loss for each sample
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

                # Check for improvement in loss1 for each sample
                improved_mask2 = (loss1 < o_best_loss1[active_mask])&(l2dist < 20.)
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                # Update the best result found so far
                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])


                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 10

        if outer_step != (binary_search_steps - 1) :
          # Create masks for successful and unsuccessful attacks
          success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
          print('success_mask ', success_mask)
          failure_mask = ~success_mask

          # Update upper bounds and constants where attacks are successful
          upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

          # Update lower bounds and constants where attacks are unsuccessful
          lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

          # Handle cases where upper bound is still too high
          upper_limit_mask = upper_bound < 1e9
          CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
          CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    # Final output based on the mask condition
    # our goal is to perturb all samples even those without successful attack
    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    #outputs = model(x_test)
    #predicted = outputs.argmax(1).unsqueeze(1)
    #acc_test = (predicted == y_test).sum().item() / len(y_test)
    #avg_acc_test.append(acc_test)

    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

# all data

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 10000
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def final_solution(oimgs, last_solution, model):
    perturbed = last_solution.clone().to(device)
    updated = oimgs.clone().to(device)
    totalchange2 = (last_solution - oimgs).abs()

    # Vectorized update of final tensor based on perturbation direction
    changes = torch.where((last_solution - oimgs) <= 0, torch.tensor(0.0, device=device), torch.tensor(1.0, device=device))

    for b in range(totalchange2.shape[0]):
        # Get sorted indices by descending change
        sorted_indices = torch.argsort(totalchange2[b], descending=True)

        for e in sorted_indices:
            if totalchange2[b, e] < 0.1:
                break

            # Update final[b, e] based on the direction of the perturbation
            updated[b, e] = changes[b, e]

            # Stop if the model's prediction is 0
            if model(updated[b].unsqueeze(0)).argmax() == 0:
                perturbed[b] = updated[b]
                break

    return perturbed

In [ ]:
#o_bestattack_rounded = final_solution(mal_x_batch, adv, combined_model)

cw_outputs = combined_model(o_bestattack_rounded)
cw_predictions = cw_outputs.argmax(1)
cw_success = (cw_predictions == 0)  # Identify succeed samples

In [ ]:
print('mean difference ',(torch.abs(o_bestattack_rounded - mal_x_batch)).sum())
print((torch.abs(o_bestattack_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
print((torch.abs(o_bestattack_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum(1))

In [ ]:
outputs = combined_model(o_bestattack_rounded)
y_pred = outputs.argmax(1)

print(outputs)
print(len(mal_y_batch))
print((y_pred == 0).sum().item())
print((y_pred == 0))
print('mean difference ',(torch.abs(o_bestattack_rounded - mal_x_batch)).sum())
print((torch.abs(o_bestattack_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())

In [ ]:
print(f"Accuracy of just malwares Under attack: {((y_pred == 1).sum().item() / len(mal_y_batch)) * 100:.4}%.")

In [ ]:
print((torch.abs(o_bestattack_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum(1))

## Part 2 — The evaluation harness

### Step 6 — Scoring rounded and non-rounded results, and checking the fixed features

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            #print(outputs)
            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            #print((y_pred == 0))
            #print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            #print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return pertb_mal_x

In [ ]:
def sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, n):
    """
    Calculate the sum of the top n smallest differences between adv_rounded and mal_x_batch
    for samples where y_pred == 0.

    Parameters:
    adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
    mal_x_batch (torch.Tensor): The original malicious examples tensor.
    y_pred (torch.Tensor): The prediction tensor.
    n (int): The number of top differences to sum.

    Returns:
    torch.Tensor: The sum of the top n smallest differences.
    """
    # Step 1: Calculate the total differences
    total_difference = torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0]).sum(1)

    # Step 2: Sort the differences in ascending order
    sorted_differences, _ = torch.sort(total_difference)

    # Step 3: Select the top n least differences
    top_n_least_differences = sorted_differences[:n]

    # Step 4: Sum these n differences
    sum_top_n_least_differences = top_n_least_differences.sum()

    return sum_top_n_least_differences

In [ ]:
def evaluate_non_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model):
    """
    Evaluate the model's performance on non-rounded adversarial examples.

    Parameters:
    - mal_x_batch: Original malware input batch
    - mal_y_batch: Ground truth labels for malware batch
    - adv: Adversarial examples generated from mal_x_batch
    - combined_model: The trained model being evaluated

    Returns:
    - A dictionary containing evaluation metrics
    """

    # Set the model to evaluation mode
    combined_model.eval()

    # Get the model's predictions on adversarial examples
    outputs = combined_model(adv)
    y_pred = outputs.argmax(dim=1)

    # Calculate metrics
    num_malwares = len(mal_y_batch)
    num_successful_attacks = (y_pred == 0).sum().item()
    total_difference = (torch.abs(adv[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
    total_difference_TOP100 = sum_of_top_n_differences(adv, mal_x_batch, y_pred, 100)
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100

    # Print the evaluation results
    print(f"Evaluation Results on NOT_Rounded Adversarial Examples")
    print(f"-----------------------------------------------")
    print(f"Number of malware samples: {num_malwares}")
    print(f"Number of successful attacks: {num_successful_attacks}")
    print(f"Total difference (L1 norm) between successful adversarial examples and original: {total_difference}")
    print(f"Total L1 norm difference - TOP 100 (successful adversarial examples vs. original): {total_difference_TOP100}")
    print(f"Accuracy of the model on malwares under attack: {accuracy:.2f}%")

    # Return results as a dictionary
    return {
        "num_malwares": num_malwares,
        "num_successful_attacks": num_successful_attacks,
        "total_difference": total_difference,
        "total_difference_TOP100": total_difference_TOP100,
        "accuracy": accuracy
    }

In [ ]:
def evaluate_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model, removal_array):
    """
    Evaluate the model's performance on rounded adversarial examples.

    Parameters:
    - mal_x_batch: Original malware input batch
    - mal_y_batch: Ground truth labels for malware batch
    - adv: Adversarial examples generated from mal_x_batch
    - combined_model: The trained model being evaluated
    - removal_array: Array used in the final rounding step

    Returns:
    - A dictionary containing evaluation metrics
    """

    # Set the model to evaluation mode
    combined_model.eval()

    # Apply the final rounding solution to adversarial examples
    adv_rounded = final_rounding(mal_x_batch, adv, combined_model, removal_array)

    # Get the model's predictions on the rounded adversarial examples
    outputs = combined_model(adv_rounded)
    y_pred = outputs.argmax(dim=1)

    # Calculate evaluation metrics
    num_malwares = len(mal_y_batch)
    num_successful_attacks = (y_pred == 0).sum().item()
    total_difference = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
    total_difference_TOP100 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 100)
    total_difference_TOP150 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 150)
    total_difference_TOP200 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 200)
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100

    # Print the evaluation results
    print(f"Evaluation Results on Rounded Adversarial Examples")
    print(f"-----------------------------------------------")
    print(f"Number of malware samples: {num_malwares}")
    print(f"Number of successful attacks: {num_successful_attacks}")
    print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
    print(f"Total L1 norm difference - TOP 100 (successful adversarial examples vs. original): {total_difference_TOP100}")
    print(f"Total L1 norm difference - TOP 150 (successful adversarial examples vs. original): {total_difference_TOP150}")
    print(f"Total L1 norm difference - TOP 200 (successful adversarial examples vs. original): {total_difference_TOP200}")
    print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

    # Return results as a dictionary
    return {
        "adv_rounded": adv_rounded,
        "num_malwares": num_malwares,
        "num_successful_attacks": num_successful_attacks,
        "total_difference": total_difference,
        "total_difference_TOP100": total_difference_TOP100,
        "total_difference_TOP150": total_difference_TOP150,
        "total_difference_TOP200": total_difference_TOP200,
        "accuracy": accuracy
    }

In [ ]:
def ensure_fixed_elements_unchanged(mal_x_batch, adv, adv_rounded, removal_array):
    """
    Ensure that fixed elements are not changed and calculate differences between adversarial examples and the original batch.

    Parameters:
    - mal_x_batch: Original malware input batch
    - adv: Adversarial examples generated from mal_x_batch
    - adv_rounded: Rounded adversarial examples
    - removal_array: Array indicating the fixed features

    Returns:
    - A dictionary containing differences for fixed and non-fixed features
    """

    # Ensure `removal_array` is expanded to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1)

    # Create a mask where non-fixed features are set to 1
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Calculate indices where the mask is 0 (fixed) or 1 (non-fixed)
    mask_fixed = non_fixed_features_mask == 0
    mask_non_fixed = non_fixed_features_mask == 1

    # Calculate the differences for fixed and non-fixed features
    diff_adv_fixed = (torch.abs(adv - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_non_fixed = (torch.abs(adv - mal_x_batch) * mask_non_fixed).sum().item()
    diff_adv_rounded_fixed = (torch.abs(adv_rounded - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_rounded_non_fixed = (torch.abs(adv_rounded - mal_x_batch) * mask_non_fixed).sum().item()

    # Print the differences
    print(f"Differences when mask is 0 (fixed features):")
    print(f"  Adv vs. Original: {diff_adv_fixed}")
    print(f"  Rounded Adv vs. Original: {diff_adv_rounded_fixed}")
    print(f"Differences when mask is 1 (non-fixed features):")
    print(f"  Adv vs. Original: {diff_adv_non_fixed}")
    print(f"  Rounded Adv vs. Original: {diff_adv_rounded_non_fixed}")

    # Return the results as a dictionary
    return {
        "diff_adv_fixed": diff_adv_fixed,
        "diff_adv_non_fixed": diff_adv_non_fixed,
        "diff_adv_rounded_fixed": diff_adv_rounded_fixed,
        "diff_adv_rounded_non_fixed": diff_adv_rounded_non_fixed
    }

In [ ]:
def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

In [ ]:
def adv_predict_rounding(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            adv_rounded = final_rounding(mal_x_batch, pertb_mal_x, model,removal_array)

            outputs = model(adv_rounded)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            #print(outputs)
            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            #print((y_pred == 0))
            #print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            #print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return pertb_mal_x

In [ ]:
def final_rounding(oimgs, newimg, model, removal_array):
    # Expand removal array to match the batch size
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8)).to(device)

    # Ensure all tensors are on the correct device
    oimgs, perturbed = oimgs.to(device), newimg.to(device)

    # Set the model to evaluation mode
    model.eval()

    # Calculate predictions and significant changes efficiently using GPU
    with torch.no_grad():
        predictions = model(oimgs).argmax(dim=1).to(device)
        active_mask = predictions == 1  # Efficiently create active mask on GPU
        total_change = (perturbed - oimgs).abs().to(device)  # Calculate absolute difference on GPU
        significant_mask = total_change > 0.1  # Determine significant changes on GPU
        sorted_idx = total_change.argsort(dim=1, descending=True)  # Sort on GPU
        changes = (perturbed > oimgs).float().to(device)

    # Copy original images to apply final rounding
    final = oimgs.clone().to(device)

    # Iterate up to the maximum number of significant changes
    # practicaly we dont have perturbs more than 120 !!
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 120)

    for i in range(max_significant_changes):
        # Safeguard against empty active mask
        if not active_mask.any():
            break

        # Get the indices of active samples
        active_indices = torch.where(active_mask)[0].to(device)

        # Get the relevant indices for current change
        indices = sorted_idx[active_indices, i]

        # since max_significant_changes is relatively large for some samples, possible that zero changes which belong to fixed_features get updated
        final[active_indices, indices] = torch.where(non_fixed_features_mask[active_indices, indices],
                                                     changes[active_indices, indices], final[active_indices, indices])

        # Update the active_mask based on the model's prediction
        with torch.no_grad():
            active_mask[active_indices] = (model(final[active_indices]).argmax(dim=1) == 1)

    return final

### Step 7 — Automatic learning-rate runs

# New Section

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import CyclicLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


def lr_range_test(model, imgs, y, removal_array, device='cuda', const=1.0, step_size_up=10000,step_size_down=10000,
                  lr_min=1e-6, lr_max=1e-1, num_iter=100, step_mode='linear', confidence=0.5):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    Parameters:
    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - lr_min: Minimum learning rate to start with.
    - lr_max: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform the LR test.
    - step_mode: 'linear' or 'exponential' mode for LR increase.
    - confidence: Confidence parameter for adversarial loss computation.

    Returns:
    - lrs: List of learning rates used during the test.
    - losses: List of corresponding losses.
    """
    model.eval()

    # Preparing tensors
    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))



    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    optimizer = torch.optim.Adam([modifier], lr=lr_min)

    scheduler = CyclicLR(optimizer, lr_min , lr_max, step_size_up=step_size_up,step_size_down=step_size_down, mode='triangular')

    def grad_hook(grad):
            return grad * non_fixed_features_mask
    modifier.register_hook(grad_hook)

    lrs = []
    losses = []
    all_success = []
    batch_size = imgs.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = imgs.clone().detach()

    for iteration in range(num_iter):
        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = sin_func_torch(modified_input)
        output = model(newimg)

        l2dist = torch.sum((newimg - imgs_tensor) ** 2, dim=1)

        real = output[torch.arange(imgs_tensor.size(0)), labs_tensor]
        other = output[torch.arange(imgs_tensor.size(0)), labs_tensor + 1]

        loss1 = torch.clamp((other - real + confidence), min=0.0)

        loss = (100 * loss1) + l2dist

        total_loss = torch.sum(loss)

        total_loss.backward()
        optimizer.step()
        scheduler.step()

        success_mask_inloop = compare(output)
        success_mask  = torch.logical_or(success_mask_inloop, success_mask)

        update_mask2 = success_mask_inloop
        o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)


        # Record learning rate and loss
        current_lr = optimizer.param_groups[0]["lr"]
        if (iteration + 1) % 1== 0:
            lrs.append(current_lr)
            losses.append(loss.detach().to('cpu'))
            all_success.append(success_mask.sum().item())

        # Debug: Print learning rate and loss
        if iteration == 0 or (iteration + 1) % 100== 0:
            current_momentum,current_momentum2 = optimizer.param_groups[0]['betas']
            print(f"Iteration {iteration + 1}: Current LR = {current_lr:.6f} \t Current momentum = {current_momentum:.6f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.6f}\t l2dist = {l2dist.sum().item():.6f}")



    return lrs, losses,o_bestattack

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

lrs, losses,adv = lr_range_test(combined_model, mal_x_batch[:1], mal_y_batch[:1], removal_array, device='cpu', const=100.0, step_size_up=1,step_size_down=10000,
                  lr_min=0.000001, lr_max=0.1, num_iter=10000, step_mode='linear', confidence=0.1)

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def torch_arctanh(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def CW(imgs, y, model, removal_array, max_iterations=1000, max_lr=1e-2,min_lr=1e-6,
       binary_search_steps=3, confidence=0.1, initial_const=1,
       cons_increase_factor=10, step_size_up=2500,step_size_down=2500):

    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    imgs_tensor = torch_arctanh(imgs).clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, min_lr , max_lr, step_size_up=step_size_up,step_size_down=step_size_down, mode='triangular')

        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            optimizer.zero_grad()

            modified_input = imgs_tensor + modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5
            output = model(newimg)

            l2dist = torch.sum((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5))** 2, dim=1)

            real = output[torch.arange(batch_size), labs_tensor]
            other = output[torch.arange(batch_size), labs_tensor + 1]

            if outer_step != (binary_search_steps - 1):
                loss1 = torch.clamp((other - real + confidence), min=0.0)
                loss = (CONST * loss1) + l2dist
            else:
                loss1 = (other - real)
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():

                success_mask_inloop = compare(output)
                success_mask  = torch.logical_or(success_mask_inloop, success_mask)

                improved_mask = (loss1 < o_best_loss1)
                o_best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)

                update_mask = (l2dist < o_bestl2) & success_mask_inloop
                o_bestl2 = torch.where(update_mask, l2dist, o_bestl2)
                o_bestattack = torch.where(update_mask.unsqueeze(-1), newimg, o_bestattack)

                if iteration == 0 or (iteration + 1) % (max_iterations // 20) == 0:
                    current_lr = scheduler.get_last_lr()[0]
                    print(f"Iteration {iteration + 1}: Current loss : {total_loss} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl2[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")


        if outer_step != (binary_search_steps - 1):
            o_success_mask  = torch.logical_or(o_success_mask, success_mask)
            print('o_success_mask ', o_success_mask.sum())

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    print(f"outer_step {outer_step + 1}: all success : {o_success_mask.sum()}")

    return final_output

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

modifier, losses,adv = lr_range_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device=device, const=100.0, step_size_up=1000,step_size_down=10000,
                  lr_min=0.000001, max_step=0.01, num_iter=10000, step_mode='linear', confidence=0.1)

## Part 3 — The systematic study

### Step 8 — The PGD-min ancestor and its final_solution

# PGD-min old version

In [ ]:
def pgd_min222(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        #loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        #loss = torch.clamp((real - other + 1.), min=0.0)
        loss = (real - other)
        #loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))

        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    #x_next = round_x(x_next, round_threshold=round_threshold)
    x_next = final_solution(x, x_next, model)

    # Compute adversarial loss
    #loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    #replace_flag = (loss_adv > loss_natural).squeeze()
    #x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

### Step 9 — Analysing the learning rate

# Analysis of Learning rate

In [ ]:
import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import CyclicLR
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import OneCycleLR

# Example model (dummy, since we only need the optimizer for this demonstration)
model = torch.nn.Linear(10, 1)

# Optimizer
optimizer = Adam(model.parameters(), lr=1e-6)

# Simulate the iterations
total_iterations = 20000  # 1 cycles


# CyclicLR Scheduler
scheduler = OneCycleLR(optimizer,
                       max_lr = 0.7, # Upper learning rate boundaries in the cycle for each parameter group
                       steps_per_epoch = 20000, # The number of steps per epoch to train for.
                       epochs = 1, # The number of epochs to train for.
                       anneal_strategy = 'cos',
                       pct_start=0.5) # Specifies the annealing strategy

'''scheduler = CyclicLR(optimizer,
                     base_lr = 1e-6, # Initial learning rate which is the lower boundary in the cycle for each parameter group
                     max_lr = 0.7, # Upper learning rate boundaries in the cycle for each parameter group
                     step_size_up = 10000, # Number of training iterations in the increasing half of a cycle
                     mode = "exp_range",
                     gamma=0.9999
                     )'''

# Store learning rates
lr_values = []


for iteration in range(total_iterations):
    # Step the scheduler to update learning rate
    scheduler.step()

    # Store the current learning rate
    lr_values.append(optimizer.param_groups[0]['lr'])

# Plotting the learning rates over iterations
plt.figure(figsize=(10, 6))
plt.plot(range(total_iterations), lr_values, label='Learning Rate')
plt.title("Learning Rate Change Over Iterations (CyclicLR)")
plt.xlabel("Iterations")
plt.ylabel("Learning Rate")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR
import math

def cyclical_lr(step, step_size_up, step_size_down, min_lr, max_lr):
    cycle_length = step_size_up + step_size_down
    cycle_position = step % cycle_length

    if cycle_position < step_size_up:
        lr = min_lr + (max_lr - min_lr) * (cycle_position / step_size_up)
    else:
        lr = max_lr - (max_lr - min_lr) * ((cycle_position - step_size_up) / step_size_down)

    return lr

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def pgd_min(x, y, model, removal_array, max_iterations, min_lr, max_lr, step_size_up=2000,step_size_down=2000, norm='l2', confidence=0.1, device=device):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()
    model.to(device)
    batch_size = x.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize starting point
    x_next = x.clone().to(device)
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8)).to(device)

    lrs = []
    losses = []

    for t in range(max_iterations):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        #loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        loss = torch.clamp((real - other + 0.001), min=0.0)
        #loss = (real - other)


        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        step_length = cyclical_lr(t, step_size_up, step_size_down, min_lr, max_lr)
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

        success_mask_inloop = compare(y_model)
        success_mask  = torch.logical_or(success_mask_inloop, success_mask)


        if t == 0 or (t + 1) % (max_iterations // 100) == 0:
          #print(f"Iteration {t + 1}: Current loss : {loss.mean().item()} \t Current success : {success_mask_inloop.sum()} \t All success : {success_mask.sum()} \t Current Learning Rate: {step_length:.3f}")
          #print(loss.detach().cpu())
          # Convert to NumPy array and print with two decimal places
          #print(np.array2string(loss.detach().cpu().numpy(), precision=2, suppress_small=True, separator=','))
          print(', '.join(f'{x:6.2f}' for x in loss.detach().cpu().numpy()))

        lrs.append(step_length)
        losses.append(loss.mean().item())

    return x_next

In [ ]:
attack_params =  {'confidence':0.001, 'removal_array':removal_array,'max_iterations': 10000,
                  'max_lr': 3., 'min_lr': 1., 'step_size_up': 2500,'step_size_down':2500}
adv2 = adv_predict(test_loader, combined_model, pgd_min, device, **attack_params)

In [ ]:
attack_params =  {'confidence':0.5, 'removal_array':removal_array,'max_iterations': 15000,
                  'max_lr': 0.5, 'min_lr': 1e-5, 'initial_const': 1 , 'binary_search_steps' : 1,
                  'cons_increase_factor' : 10, 'step_size_up': 1000,'step_size_down':1000}
adv = adv_predict(test_loader, combined_model, CW, device, **attack_params)

### Step 10 — The learning-rate range test

# PGD-min:  loss=(real - other)

## Boundary of learning rate

In [ ]:
def LR(lr_min, lr_max, step_mode, num_iter, iteration):
      # Define learning rate schedule
    if step_mode == 'linear':
        lr =  lr_min + (iteration / (num_iter - 1)) * (lr_max - lr_min)
    elif step_mode == 'exponential':
        lr = lr_min * (lr_max / lr_min) ** (iteration / (num_iter - 1))
    else:
        raise ValueError("Invalid step_mode. Choose 'linear' or 'exponential'.")
    return lr

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def lr_range_test(model, x, y, removal_array, num_iter, lr_min, lr_max, step_mode, norm='l2', confidence=0.1, device='cuda'):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()
    model.to(device)



    # Initialize starting point
    x_next = x.clone().to(device)
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8)).to(device)

    lrs = []
    losses = []
    all_success = []
    batch_size = x.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = x.clone().detach()

    for t in range(num_iter):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)

        y_model = model(x_var)

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        #loss = torch.clamp((real - other + confidence), min=0.0)
        loss = (real - other)

        with torch.no_grad():
            perturbation = torch.where((x==1.), (x - x_next), (x_next - x))
            l2dist = torch.sum(perturbation, dim=1)

            success_mask_inloop = compare(y_model)
            success_mask  = torch.logical_or(success_mask_inloop, success_mask)

            update_mask2 = success_mask_inloop
            o_bestattack = torch.where(update_mask2.unsqueeze(-1), x_var, o_bestattack)


        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        step_length = LR(lr_min, lr_max, step_mode, num_iter, t)
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

        lrs.append(step_length)
        losses.append(loss.mean().item())
        all_success.append(success_mask.sum().item())

        # Debug: Print learning rate and loss
        if t == 0 or (t + 1) % 1000 == 0:
            print(f"Iteration {t + 1}/{num_iter}: Current LR = {step_length:.6f}, success = {success_mask.sum().item()}, Loss = {loss.mean().item():.6f}")


    # Plot the LR Range Test results using seaborn
    sns.set(style="whitegrid")
    plt.figure(figsize=(40, 10))

    # Plot the loss versus learning rate
    sns.lineplot(x=lrs, y=all_success)

    # Calculate the interval for showing only 5% of the learning rates
    interval = max(1, len(lrs) // 40)  # 5% of the learning rates

    # Sample the learning rates and corresponding positions
    sampled_lrs = lrs[::interval]
    sampled_labels = [f'{lr:.2e}' for lr in sampled_lrs]

    # Set the ticks to be the sampled learning rates
    plt.xticks(ticks=sampled_lrs, labels=sampled_labels, rotation=45)

    plt.xlabel('Learning Rate')
    plt.ylabel('Loss')
    plt.title('LR Range Test')

    # Enable grid and show the plot
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.show()

    return lrs, all_success,o_bestattack

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

lrs, all_success,adv = lr_range_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device=device,
                  lr_min=1e-6, lr_max=6, num_iter=10000, step_mode='linear', confidence=0.1)

In [ ]:
lrs, all_success,adv = lr_range_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device=device,
                  lr_min=1e-6, lr_max=6, num_iter=30000, step_mode='linear', confidence=0.1)

### Step 11 — Rounding at the end vs in the loop (one constraint)

## Rounded at the end - one const

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR
import math

def cyclical_lr(step, step_size_up, step_size_down, min_lr, max_lr):
    cycle_length = step_size_up + step_size_down
    cycle_position = step % cycle_length

    if cycle_position < step_size_up:
        lr = min_lr + (max_lr - min_lr) * (cycle_position / step_size_up)
    else:
        lr = max_lr - (max_lr - min_lr) * ((cycle_position - step_size_up) / step_size_down)

    return lr

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def pgd_min(x, y, model, removal_array, max_iterations, min_lr, max_lr, step_size_up=2000,step_size_down=2000, norm='l2', confidence=0.1, device=device):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()
    model.to(device)



    # Initialize starting point
    x_next = x.clone().to(device)
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8)).to(device)

    lrs = []
    losses = []
    batch_size = x.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = x.clone().detach()
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)

    for t in range(max_iterations):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)

        y_model = model(x_var)

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        #loss = torch.clamp((real - other + confidence), min=0.0)
        loss = (real - other)

        with torch.no_grad():
            perturbation = torch.where((x==1.), (x - x_next), (x_next - x))
            l2dist = torch.sum(perturbation, dim=1)

            success_mask_inloop = compare(y_model)
            success_mask  = torch.logical_or(success_mask_inloop, success_mask)

            update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
            o_bestattack = torch.where(update_mask2.unsqueeze(-1), x_var, o_bestattack)

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        step_length = cyclical_lr(t, step_size_up, step_size_down, min_lr, max_lr)
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

        if t == 0 or (t + 1) % (max_iterations // 20) == 0:
          print(f"Iteration {t + 1}/{max_iterations}: Current LR = {step_length:.6f}, success = {success_mask.sum().item()}, Loss = {loss.mean().item():.6f}")

        lrs.append(step_length)
        losses.append(loss.mean().item())

    return o_bestattack

In [ ]:
attack_params =  {'confidence':0.1, 'removal_array':removal_array,'max_iterations': 30000,
                  'max_lr': 6, 'min_lr': 1e-6, 'step_size_up': 1,'step_size_down':29999}
adv = adv_predict(test_loader, combined_model, pgd_min, device, **attack_params)

In [ ]:
results_non_rounded = evaluate_non_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model)
print(' ')
results_rounded = evaluate_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model, removal_array)

## Rounded inloop - one const

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR
import math

def cyclical_lr(step, step_size_up, step_size_down, min_lr, max_lr):
    cycle_length = step_size_up + step_size_down
    cycle_position = step % cycle_length

    if cycle_position < step_size_up:
        lr = min_lr + (max_lr - min_lr) * (cycle_position / step_size_up)
    else:
        lr = max_lr - (max_lr - min_lr) * ((cycle_position - step_size_up) / step_size_down)

    return lr

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def pgd_min_l1(x, y, model, removal_array, max_iterations, min_lr, max_lr, step_size_up=2000,step_size_down=2000, norm='l2', confidence=0.1, device=device):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()
    model.to(device)



    # Initialize starting point
    x_next = x.clone().to(device)
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8)).to(device)

    lrs = []
    losses = []
    batch_size = x.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestl0 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = x.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = x.clone().detach()

    for t in range(max_iterations):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)

        y_model = model(x_var)

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        #loss = torch.clamp((real - other + confidence), min=0.0)
        loss = (real - other)

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        step_length = cyclical_lr(t, step_size_up, step_size_down, min_lr, max_lr)
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

        if (t + 1) % 10 == 0:
            with torch.no_grad():

                success_mask_inloop, l0dist = compare_round(x,x_var, model,removal_array)
                success_mask  = torch.logical_or(success_mask_inloop, success_mask)

                improved_mask = (loss < o_best_loss1)
                o_best_loss1 = torch.where(improved_mask, loss, o_best_loss1)
                o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), x_next, o_best_unsuccessfull_attack)

                update_mask = (l0dist < o_bestl0) & success_mask_inloop
                o_bestl0 = torch.where(update_mask, l0dist, o_bestl0)
                o_bestattack = torch.where(update_mask.unsqueeze(-1), x_next, o_bestattack)

                if (t + 1) % 500 == 0:
                    print(f"Iteration {t + 1}: Current loss : {loss.sum().item()} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl0[success_mask].mean():.6f} \t Learning Rate: {step_length:.3f}")

        lrs.append(step_length)
        losses.append(loss.mean().item())

    print('**************************************')
    print(f"FINAL: success = {success_mask.sum().item()}, o_bestl0 = {o_bestl0[success_mask].mean().item():.6f}")
    return o_bestattack

In [ ]:
attack_params =  {'confidence':0.1, 'removal_array':removal_array,'max_iterations': 30000,
                  'max_lr': 6, 'min_lr': 1e-6, 'step_size_up':15000,'step_size_down':15000}
adv = adv_predict_rounding(test_loader, combined_model, pgd_min_l1, device, **attack_params)

### Step 12 — The L2 distance: range test and rounding variants

# dist : L2 norm

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import CyclicLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


def lr_range_test(model, imgs, y, removal_array, device='cuda', const=1.0, step_size_up=10000,step_size_down=10000,
                  lr_min=1e-6, lr_max=1e-1, num_iter=100, step_mode='linear', confidence=0.5):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    Parameters:
    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - lr_min: Minimum learning rate to start with.
    - lr_max: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform the LR test.
    - step_mode: 'linear' or 'exponential' mode for LR increase.
    - confidence: Confidence parameter for adversarial loss computation.

    Returns:
    - lrs: List of learning rates used during the test.
    - losses: List of corresponding losses.
    """
    model.eval()

    # Preparing tensors
    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))



    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    optimizer = torch.optim.Adam([modifier], lr=lr_min)

    scheduler = CyclicLR(optimizer, lr_min , lr_max, step_size_up=step_size_up,step_size_down=step_size_down, mode='triangular2', base_momentum=0.5, max_momentum=0.9)

    def grad_hook(grad):
            return grad * non_fixed_features_mask
    modifier.register_hook(grad_hook)

    lrs = []
    losses = []
    all_success = []
    batch_size = imgs.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = imgs.clone().detach()

    for iteration in range(num_iter):
        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = sin_func_torch(modified_input)
        output = model(newimg)

        l2dist = torch.sum((newimg - imgs_tensor) ** 2, dim=1)

        real = output[torch.arange(imgs_tensor.size(0)), labs_tensor]
        other = output[torch.arange(imgs_tensor.size(0)), labs_tensor + 1]

        loss1 = (other - real )
        loss = torch.sum(loss1)

        loss.backward()
        optimizer.step()
        scheduler.step()

        success_mask_inloop = compare(output)
        success_mask  = torch.logical_or(success_mask_inloop, success_mask)

        update_mask2 = success_mask_inloop
        o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)



        # Record learning rate and loss
        current_lr = optimizer.param_groups[0]["lr"]
        lrs.append(current_lr)
        losses.append(loss.item())
        all_success.append(success_mask.sum().item())

        # Debug: Print learning rate and loss
        if iteration == 0 or (iteration + 1) % 100== 0:
            current_momentum,current_momentum2 = optimizer.param_groups[0]['betas']
            print(f"Iteration {iteration + 1}: Current LR = {current_lr:.6f} \t Current momentum = {current_momentum:.6f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {loss.item():.6f}")



    return lrs, losses,o_bestattack

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

lrs, losses,adv = lr_range_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', const=100.0, step_size_up=5000,step_size_down=5000,
                  lr_min=1e-6, lr_max=0.02, num_iter=40000, step_mode='linear', confidence=0.1)

## Rounded at the end - one const

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW(imgs, y, model, removal_array, max_iterations=1000, max_lr=1e-2,min_lr=1e-6,
       binary_search_steps=3, confidence=0.1, initial_const=1,
       cons_increase_factor=10, step_size_up=2500,step_size_down=2500):

    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, min_lr , max_lr, step_size_up=step_size_up,step_size_down=step_size_down, mode='triangular')

        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            optimizer.zero_grad()

            modified_input = imgs_tensor + modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor) ** 2, dim=1)

            real = output[torch.arange(batch_size), labs_tensor]
            other = output[torch.arange(batch_size), labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)

            loss = (CONST * loss1) + l2dist

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():

                success_mask_inloop = compare(output)
                success_mask  = torch.logical_or(success_mask_inloop, success_mask)

                improved_mask = (loss1 < o_best_loss1)
                o_best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)

                update_mask = (l2dist < o_bestl2) & success_mask_inloop
                o_bestl2 = torch.where(update_mask, l2dist, o_bestl2)
                o_bestattack = torch.where(update_mask.unsqueeze(-1), newimg, o_bestattack)

                if iteration == 0 or (iteration + 1) % (max_iterations // 20) == 0:
                    current_lr = scheduler.get_last_lr()[0]
                    print(f"Iteration {iteration + 1}: Current loss : {total_loss} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl2[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")


        if outer_step != (binary_search_steps - 1):
            o_success_mask  = torch.logical_or(o_success_mask, success_mask)
            print('o_success_mask ', o_success_mask.sum())

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    print(f"outer_step {outer_step + 1}: all success : {o_success_mask.sum()}")

    return final_output

In [ ]:
attack_params =  {'confidence':0.1, 'removal_array':removal_array,'max_iterations': 30000,
                  'max_lr': 0.7, 'min_lr': 1e-6, 'initial_const': 10 , 'binary_search_steps' : 1,
                  'cons_increase_factor' : 10, 'step_size_up': 1,'step_size_down':29999}
adv = adv_predict(test_loader, combined_model, CW, device, **attack_params)

### Step 13 — All constraints: rounding at the end, in the loop, and the tanh variant

## Rounded at the end - all const

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW(imgs, y, model, removal_array, max_iterations=1000, max_lr=1e-2,min_lr=1e-6,
       binary_search_steps=3, confidence=0.1, initial_const=1,
       cons_increase_factor=10, step_size_up=2500,step_size_down=2500):

    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, min_lr , max_lr, step_size_up=step_size_up,step_size_down=step_size_down, mode='triangular')

        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            optimizer.zero_grad()

            modified_input = imgs_tensor + modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor) ** 2, dim=1)

            real = output[torch.arange(batch_size), labs_tensor]
            other = output[torch.arange(batch_size), labs_tensor + 1]

            if outer_step != (binary_search_steps - 1):
                loss1 = torch.clamp((other - real + confidence), min=0.0)
                loss = (CONST * loss1) + l2dist
            else:
                loss1 = (other - real)
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():

                success_mask_inloop = compare(output)
                success_mask  = torch.logical_or(success_mask_inloop, success_mask)

                improved_mask = (loss1 < o_best_loss1)
                o_best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)

                update_mask = (l2dist < o_bestl2) & success_mask_inloop
                o_bestl2 = torch.where(update_mask, l2dist, o_bestl2)
                o_bestattack = torch.where(update_mask.unsqueeze(-1), newimg, o_bestattack)

                if iteration == 0 or (iteration + 1) % (max_iterations // 20) == 0:
                    current_lr = scheduler.get_last_lr()[0]
                    print(f"Iteration {iteration + 1}: Current loss : {total_loss} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl2[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")


        if outer_step != (binary_search_steps - 1):
            o_success_mask  = torch.logical_or(o_success_mask, success_mask)
            print('o_success_mask ', o_success_mask.sum())

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    print(f"outer_step {outer_step + 1}: all success : {o_success_mask.sum()}")

    return final_output

In [ ]:
attack_params =  {'confidence':0.1, 'removal_array':removal_array,'max_iterations': 30000,
                  'max_lr': 0.7, 'min_lr': 1e-6, 'initial_const': 1 , 'binary_search_steps' : 6,
                  'cons_increase_factor' : 10, 'step_size_up': 10000,'step_size_down':10000}
adv = adv_predict(test_loader, combined_model, CW, device, **attack_params)

## Rounded at the end - all const - tanh

## Rounded inloop - all const

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, max_lr=1e-2,min_lr=1e-6,
       binary_search_steps=3, confidence=0.1, initial_const=1,
       cons_increase_factor=10, step_size_up=2500,step_size_down=2500):

    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl0 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, min_lr , max_lr, step_size_up=step_size_up,step_size_down=step_size_down, mode='triangular2')

        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            optimizer.zero_grad()

            modified_input = imgs_tensor + modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor) ** 2, dim=1)

            real = output[torch.arange(batch_size), labs_tensor]
            other = output[torch.arange(batch_size), labs_tensor + 1]

            if outer_step != (binary_search_steps - 1):
                loss1 = torch.clamp((other - real + confidence), min=0.0)
                loss = (CONST * loss1) + l2dist
            else:
                loss1 = (other - real)
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            if (iteration + 1) % 20 == 0:
                with torch.no_grad():

                    success_mask_inloop, l0dist = compare_round(imgs_tensor,newimg, model,removal_array)
                    success_mask  = torch.logical_or(success_mask_inloop, success_mask)

                    improved_mask = (loss1 < o_best_loss1)
                    o_best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                    o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)

                    update_mask = (l0dist < o_bestl0) & success_mask_inloop
                    o_bestl0 = torch.where(update_mask, l0dist, o_bestl0)
                    o_bestattack = torch.where(update_mask.unsqueeze(-1), newimg, o_bestattack)

                    if (iteration + 1) % 500 == 0:
                        current_lr = scheduler.get_last_lr()[0]
                        print(f"Iteration {iteration + 1}: Current loss : {total_loss} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl0[success_mask].mean():.6f} \t Learning Rate: {current_lr:.3f}")


        if outer_step != (binary_search_steps - 1):
            #o_success_mask  = torch.logical_or(o_success_mask, success_mask)
            #print('o_success_mask ', o_success_mask.sum())
            o_success_attack, o_l0dist = compare_round(imgs,o_bestattack, model,removal_array)
            print(f"outer_step {outer_step + 1}:  all success : {o_success_attack.sum()} \t mean(l0)(success) : {o_l0dist[o_success_attack].mean():.4f}")


            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl0 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    print(f"outer_step {outer_step + 1}: all success : {o_success_mask.sum()}")

    return final_output

In [ ]:
attack_params =  {'confidence':0.1, 'removal_array':removal_array,'max_iterations': 20000,
                  'max_lr': 0.7, 'min_lr': 1e-6, 'initial_const': 10 , 'binary_search_steps' : 5,
                  'cons_increase_factor' : 10, 'step_size_up': 10000,'step_size_down':10000}
adv = adv_predict_rounding(test_loader, combined_model, CW_l1, device, **attack_params)

### Step 14 — Whole-tensor rounding

## Rounded inloop - all const(new)

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import OneCycleLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, max_lr=1e-2,min_lr=1e-6,
       binary_search_steps=3, confidence=0.1, initial_const=1,
       cons_increase_factor=10):

    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl0 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = OneCycleLR(optimizer,
                              max_lr = max_lr, # Upper learning rate boundaries in the cycle for each parameter group
                              steps_per_epoch = max_iterations, # The number of steps per epoch to train for.
                              epochs = 1, # The number of epochs to train for.
                              anneal_strategy = 'linear',
                              base_momentum=0.5, max_momentum=0.95,
                              pct_start=0.5) # Specifies the annealing strategy

        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            optimizer.zero_grad()

            modified_input = imgs_tensor + modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor) ** 2, dim=1)

            real = output[torch.arange(batch_size), labs_tensor]
            other = output[torch.arange(batch_size), labs_tensor + 1]

            if outer_step != (binary_search_steps - 1):
                loss1 = torch.clamp((other - real + confidence), min=0.0)
                loss = (CONST * loss1) + l2dist
            else:
                loss1 = (other - real)
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            if (iteration + 1) % 25 == 0:
                with torch.no_grad():

                    success_mask_inloop, l0dist = compare_round(imgs_tensor,newimg, model,removal_array)
                    success_mask  = torch.logical_or(success_mask_inloop, success_mask)

                    improved_mask = (loss1 < o_best_loss1)
                    o_best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                    o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)

                    update_mask = (l0dist < o_bestl0) & success_mask_inloop
                    o_bestl0 = torch.where(update_mask, l0dist, o_bestl0)
                    o_bestattack = torch.where(update_mask.unsqueeze(-1), newimg, o_bestattack)

                    if (iteration + 1) % 500 == 0:
                        current_lr = scheduler.get_last_lr()[0]
                        current_momentum,current_momentum2 = optimizer.param_groups[0]['betas']
                        print(f"Iteration {iteration + 1:5d}:\t Current loss : {total_loss:.2f} \t Current success : {success_mask_inloop.sum()} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl0[success_mask].mean():.4f} \t Current LR: {current_lr:.4f}\t Current Momentum: {current_momentum:.4f}")


        if outer_step != (binary_search_steps - 1):
            #o_success_mask  = torch.logical_or(o_success_mask, success_mask)
            #print('o_success_mask ', o_success_mask.sum())
            o_success_attack, o_l0dist = compare_round(imgs,o_bestattack, model,removal_array)
            print(f"outer_step {outer_step + 1}:  all success : {o_success_attack.sum()} \t mean(l0)(success) : {o_l0dist[o_success_attack].mean():.4f}")


            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl0 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    print(f"outer_step {outer_step + 1}: all success : {o_success_mask.sum()}")

    return final_output

In [ ]:
attack_params =  {'confidence':0.1, 'removal_array':removal_array,'max_iterations': 40000,
                  'max_lr': 0.035, 'min_lr': 1e-5, 'initial_const': 10 , 'binary_search_steps' : 6,
                  'cons_increase_factor' : 10}
adv = adv_predict_rounding(test_loader, combined_model, CW_l1, device, **attack_params)

### Step 15 — The L1 distance, and the tanh distance

# dist : L1 norm

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


def lr_range_test(model, imgs, y, removal_array, device='cuda', const=1.0,
                  lr_min=1e-6, lr_max=1e-1, num_iter=100, step_mode='linear', confidence=0.5):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    Parameters:
    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - lr_min: Minimum learning rate to start with.
    - lr_max: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform the LR test.
    - step_mode: 'linear' or 'exponential' mode for LR increase.
    - confidence: Confidence parameter for adversarial loss computation.

    Returns:
    - lrs: List of learning rates used during the test.
    - losses: List of corresponding losses.
    """
    model.eval()

    # Preparing tensors
    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))



    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    optimizer = torch.optim.Adam([modifier], lr=lr_min)

    # Define learning rate schedule
    if step_mode == 'linear':
        lr_lambda = lambda iteration: 1 + (iteration / (num_iter - 1)) * (lr_max / lr_min - 1)
    elif step_mode == 'exponential':
        lr_lambda = lambda iteration: (lr_max / lr_min) ** (iteration / (num_iter - 1))
    else:
        raise ValueError("Invalid step_mode. Choose 'linear' or 'exponential'.")

    scheduler = LambdaLR(optimizer, lr_lambda)

    def grad_hook(grad):
            return grad * non_fixed_features_mask
    modifier.register_hook(grad_hook)

    lrs = []
    losses = []
    all_success = []
    batch_size = imgs.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = imgs.clone().detach()

    for iteration in range(num_iter):
        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = sin_func_torch(modified_input)
        output = model(newimg)

        perturbation = torch.where((imgs_tensor==1.), (imgs_tensor - newimg), (newimg - imgs_tensor))
        l2dist = torch.sum(perturbation, dim=1)

        real = output[torch.arange(imgs_tensor.size(0)), labs_tensor]
        other = output[torch.arange(imgs_tensor.size(0)), labs_tensor + 1]

        loss1 = torch.clamp(other - real + confidence, min=0.0)
        loss = torch.sum(const * loss1 + l2dist)

        loss.backward()
        optimizer.step()
        scheduler.step()

        success_mask_inloop = compare(output)
        success_mask  = torch.logical_or(success_mask_inloop, success_mask)

        update_mask2 = success_mask_inloop
        o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)



        # Record learning rate and loss
        current_lr = optimizer.param_groups[0]["lr"]
        lrs.append(current_lr)
        losses.append(loss.item())
        all_success.append(success_mask.sum().item())

        # Debug: Print learning rate and loss
        if iteration == 0 or (iteration + 1) % (num_iter // 10) == 0:
            print(f"Iteration {iteration + 1}/{num_iter}: Current LR = {current_lr:.6f}, success = {success_mask.sum().item()}, Loss = {loss.item():.6f}")

    # Plot the LR Range Test results using seaborn
    sns.set(style="whitegrid")
    plt.figure(figsize=(40, 10))

    # Plot the loss versus learning rate
    sns.lineplot(x=lrs, y=all_success)



    # Calculate the interval for showing only 5% of the learning rates
    interval = max(1, len(lrs) // 40)  # 5% of the learning rates

    # Sample the learning rates and corresponding positions
    sampled_lrs = lrs[::interval]
    sampled_labels = [f'{lr:.2e}' for lr in sampled_lrs]

    # Set the ticks to be the sampled learning rates
    plt.xticks(ticks=sampled_lrs, labels=sampled_labels, rotation=45)

    plt.xlabel('Learning Rate')
    plt.ylabel('Loss')
    plt.title('LR Range Test')

    # Enable grid and show the plot
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.show()


    return lrs, all_success,o_bestattack

In [ ]:
lrs, all_success,adv = lr_range_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', const=10.0,
                  lr_min=1e-6, lr_max=1.5, num_iter=30000, step_mode='linear', confidence=0.1)

# dist : tanh(2 * perturbation)

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


def lr_range_test(model, imgs, y, removal_array, device='cuda', const=1.0,
                  lr_min=1e-6, lr_max=1e-1, num_iter=100, step_mode='linear', confidence=0.5):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    Parameters:
    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - lr_min: Minimum learning rate to start with.
    - lr_max: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform the LR test.
    - step_mode: 'linear' or 'exponential' mode for LR increase.
    - confidence: Confidence parameter for adversarial loss computation.

    Returns:
    - lrs: List of learning rates used during the test.
    - losses: List of corresponding losses.
    """
    model.eval()

    # Preparing tensors
    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))



    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    optimizer = torch.optim.Adam([modifier], lr=lr_min)

    # Define learning rate schedule
    if step_mode == 'linear':
        lr_lambda = lambda iteration: 1 + (iteration / (num_iter - 1)) * (lr_max / lr_min - 1)
    elif step_mode == 'exponential':
        lr_lambda = lambda iteration: (lr_max / lr_min) ** (iteration / (num_iter - 1))
    else:
        raise ValueError("Invalid step_mode. Choose 'linear' or 'exponential'.")

    scheduler = LambdaLR(optimizer, lr_lambda)

    def grad_hook(grad):
            return grad * non_fixed_features_mask
    modifier.register_hook(grad_hook)

    lrs = []
    losses = []
    all_success = []
    batch_size = imgs.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = imgs.clone().detach()

    for iteration in range(num_iter):
        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = sin_func_torch(modified_input)
        output = model(newimg)

        perturbation = torch.where((imgs_tensor==1.), (imgs_tensor - newimg), (newimg - imgs_tensor))
        l2dist = torch.sum(torch.tanh(2 * perturbation), dim=1)

        real = output[torch.arange(imgs_tensor.size(0)), labs_tensor]
        other = output[torch.arange(imgs_tensor.size(0)), labs_tensor + 1]

        loss1 = torch.clamp(other - real + confidence, min=0.0)
        loss = torch.sum(const * loss1 + l2dist)

        loss.backward()
        optimizer.step()
        scheduler.step()

        success_mask_inloop = compare(output)
        success_mask  = torch.logical_or(success_mask_inloop, success_mask)

        update_mask2 = success_mask_inloop
        o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)



        # Record learning rate and loss
        current_lr = optimizer.param_groups[0]["lr"]
        lrs.append(current_lr)
        losses.append(loss.item())
        all_success.append(success_mask.sum().item())

        # Debug: Print learning rate and loss
        if iteration == 0 or (iteration + 1) % (num_iter // 10) == 0:
            print(f"Iteration {iteration + 1}/{num_iter}: Current LR = {current_lr:.6f}, success = {success_mask.sum().item()}, Loss = {loss.item():.6f}")

    # Plot the LR Range Test results using seaborn
    sns.set(style="whitegrid")
    plt.figure(figsize=(40, 10))

    # Plot the loss versus learning rate
    sns.lineplot(x=lrs, y=all_success)



    # Calculate the interval for showing only 5% of the learning rates
    interval = max(1, len(lrs) // 40)  # 5% of the learning rates

    # Sample the learning rates and corresponding positions
    sampled_lrs = lrs[::interval]
    sampled_labels = [f'{lr:.2e}' for lr in sampled_lrs]

    # Set the ticks to be the sampled learning rates
    plt.xticks(ticks=sampled_lrs, labels=sampled_labels, rotation=45)

    plt.xlabel('Learning Rate')
    plt.ylabel('Loss')
    plt.title('LR Range Test')

    # Enable grid and show the plot
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.show()


    return lrs, all_success,o_bestattack

In [ ]:
# l2dist = torch.sum(torch.tanh(2 * perturbation), dim=1)
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

lrs, all_success,adv = lr_range_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', const=10.0,
                  lr_min=1e-6, lr_max=1, num_iter=10000, step_mode='linear', confidence=0.1)

## Rounded at the end - one const

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += confidence  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW(imgs, y, model, removal_array, max_iterations=1000, max_lr=1e-2,min_lr=1e-6,
       binary_search_steps=3, confidence=0.1, initial_const=1,
       cons_increase_factor=10, step_size_up=2500,step_size_down=2500):

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, min_lr , max_lr, step_size_up=step_size_up,step_size_down=step_size_down, mode='triangular')


        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            perturbation = torch.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor))
            l2dist = torch.sum(torch.tanh(2 * perturbation), dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)

            loss = (active_CONST * loss1) + l2dist

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():

                success_mask_inloop = compare(output, active_labs_tensor, confidence, targeted)
                success_mask  = torch.logical_or(success_mask_inloop, success_mask)

                improved_mask2 = (loss1 < o_best_loss1[active_mask])
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                update_mask = (l2dist < bestl2[active_mask]) & success_mask_inloop
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & success_mask_inloop
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                if iteration == 0 or (iteration + 1) % (max_iterations // 10) == 0:
                    current_lr = scheduler.get_last_lr()[0]
                    print(f"Iteration {iteration + 1}: Current loss : {total_loss} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl2[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")


        if outer_step != (binary_search_steps - 1):
            o_success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
            o_success_attack  = torch.logical_or(o_success_attack, o_success_mask)
            print('o_success_attack ', o_success_attack.sum())

            failure_mask = ~o_success_mask
            upper_bound[o_success_mask] = torch.minimum(upper_bound[o_success_mask], CONST[o_success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    print(f"outer_step {outer_step + 1}: all success : {o_success_attack.sum()}")

    return final_output

In [ ]:
attack_params =  {'confidence':0.1, 'removal_array':removal_array,'max_iterations': 20000,
                  'max_lr': 0.9, 'min_lr': 1e-6, 'initial_const': 10 , 'binary_search_steps' : 1,
                  'cons_increase_factor' : 10, 'step_size_up': 1,'step_size_down':19999}
adv = adv_predict(test_loader, combined_model, CW, device, **attack_params)

## Part 4 — The final form (CW_sine5)

### Step 16 — Final rounding, in two versions

In [ ]:
def final_rounding(oimgs, newimg, model):
    # Ensure all tensors are on the GPU
    oimgs = oimgs.to(device)
    perturbed = newimg.to(device)

    # Set the model to evaluation mode
    model.eval()

    # Initialize active_mask based on model's initial prediction
    active_mask = (model(oimgs).argmax(dim=1) == 1)

    # Calculate the absolute difference between original and perturbed images
    total_change = (perturbed - oimgs).abs()

    # Determine which changes are significant
    significant_mask = total_change > 0.1

    # Prepare a copy of the original images to apply the final rounding
    final = oimgs.clone()

    # Determine the direction of changes: 0 for decrease/no change, 1 for increase
    changes = (perturbed > oimgs).float()

    # Get sorted indices of the total changes for each image in descending order
    sorted_idx = total_change.argsort(dim=1, descending=True)

    # Get the maximum number of significant changes across all images
    max_significant_changes = significant_mask.sum(dim=1).max()

    # Iterate over each significant change index
    for i in range(max_significant_changes):
        # Update active_mask indices
        active_indices = torch.nonzero(active_mask).squeeze()

        if active_indices.numel() == 0:
            break

        # Safeguard when active_indices is a single-element tensor
        if active_indices.dim() == 0:
            active_indices = active_indices.unsqueeze(0)

        # Get the relevant indices for current change
        indices = sorted_idx[active_indices, i]

        # Apply the changes to the final image tensor
        final[active_indices, indices] = changes[active_indices, indices]

        # Update the active_mask based on the model's output
        active_mask[active_indices] = (model(final[active_indices]).argmax(dim=1) == 1)

    return final

# New function instead of tanh

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import newton

# Define the function
def f(x):
    return 3 * x**2 - 2 * x**3

# Generate values for x from 0 to 1
x_values = np.linspace(-0.5, 1.5, 500)
y_values = f(x_values)

# Plot the function
plt.figure(figsize=(10, 6))
plt.plot(x_values, y_values, label=r"$f(x) = 3x^2 - 2x^3$", color="blue")

# Add labels and title
plt.title("Plot of the Function $f(x) = 3x^2 - 2x^3$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.grid(True)
plt.legend()
plt.show()

sine function

In [ ]:
# Define the function and its gradient
def sin_func(x):
    return (np.sin((x-0.5)*(np.pi))+1)/2.

# its derivative
def sin_func_prime(x):
    return np.pi/2. * np.cos((x-0.5)*(np.pi))

# Generate values for x from -2π to 2π
x_values = np.linspace(-0.5, 1.5, 1000)
sin_values = sin_func(x_values)
sin_prime_values = sin_func_prime(x_values)

# Plot sin(x)
plt.figure(figsize=(10, 6))
plt.plot(x_values, sin_values, label=r"$\sin(x)$", color="blue")

# Add labels and title
plt.title("Plot of $\sin(x)$")
plt.xlabel("x")
plt.ylabel("$\sin(x)$")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.grid(True)
plt.legend()
plt.show()

# Plot the gradient of sin(x)
plt.figure(figsize=(10, 6))
plt.plot(x_values, sin_prime_values, label=r"$\cos(x)$", color="red")

# Add labels and title
plt.title("Plot of the Gradient of $\sin(x)$: $\cos(x)$")
plt.xlabel("x")
plt.ylabel("$\cos(x)$")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
print(sin_func(0))
print(sin_func(1))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the function
def sin_func(x):
    return (np.sin((x-0.5)*(np.pi))+1)/2.

# Generate values for x including 0 and 1 explicitly
x_values = np.linspace(-0.5, 1.5, 10000)

# Ensure 0 and 1 are in the list by adding them and sorting the array
x_values = np.sort(np.unique(np.concatenate((x_values, [0, 1]))))

sin_values = sin_func(x_values)

# Calculate the gradient (derivative) numerically using np.gradient
sin_prime_values = np.gradient(sin_values, x_values)

# Create a figure with two subplots
fig, axs = plt.subplots(2, 1, figsize=(10, 10))

# Plot sin(x) in the first subplot
axs[0].plot(x_values, sin_values, label=r"$\sin(x)$", color="blue")
axs[0].set_title("Plot of $\sin(x)$")
axs[0].set_xlabel("x")
axs[0].set_ylabel(r"$\sin(x)$")
axs[0].axhline(0, color='black', linewidth=0.5)
axs[0].axvline(0, color='black', linewidth=0.5)
axs[0].grid(True)
axs[0].legend()

# Plot the gradient of sin(x) in the second subplot
axs[1].plot(x_values, sin_prime_values, label=r"$\frac{d}{dx}\sin(x)$", color="red")
axs[1].set_title("Plot of Gradient of $\sin(x)$")
axs[1].set_xlabel("x")
axs[1].set_ylabel(r"$\frac{d}{dx}\sin(x)$")
axs[1].axhline(0, color='black', linewidth=0.5)
axs[1].axvline(0, color='black', linewidth=0.5)
axs[1].grid(True)
axs[1].legend()

# Show the plots
plt.tight_layout()
plt.show()

**consider that we use "Numerical Differentiation"(estimation) not exact values for derivatives**

# TEST

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += confidence  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

# Define the DataLoader for training, validation, and test sets
batch_size = 10000
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    #outputs = model(x_test)
    #predicted = outputs.argmax(1).unsqueeze(1)
    #acc_test = (predicted == y_test).sum().item() / len(y_test)
    #avg_acc_test.append(acc_test)

    imgs, y = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

model = model_AT_rFGSM
max_iterations=5000
learning_rate=0.01
binary_search_steps=1
confidence=0.1
initial_const=10
cons_increase_factor=10

labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
batch_size = imgs.shape[0]
targeted = True

# Expand removal array to match the batch size
expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

mask_0 = non_fixed_features_mask == 0
mask_1 = non_fixed_features_mask == 1

imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

o_bestl2 = torch.full((batch_size,), 1e10, device=device)
o_bestscore = torch.full((batch_size,), -1, device=device)
o_bestattack = imgs.clone().detach()

o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
o_best_unsuccessfull_attack = imgs.clone().detach()
o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

model.eval()

for outer_step in range(binary_search_steps):
    print(CONST)

    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    optimizer = torch.optim.Adam([modifier], lr=learning_rate)

    def grad_hook(grad):
        return grad * non_fixed_features_mask
    modifier.register_hook(grad_hook)

    scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=0.1, step_size_up=4,
                          step_size_down=4995, mode='triangular2')

    best_loss = torch.full((batch_size,), float('inf'), device=device)
    no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
    active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

    bestl2 = torch.full((batch_size,), 1e10, device=device)
    bestscore = torch.full((batch_size,), -1, device=device)

    for iteration in range(max_iterations):
        if (iteration + 1) % 100 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"Iteration {iteration + 1}: Current Learning Rate: {current_lr}")

        active_indices = torch.where(active_mask)[0]
        active_modifier = modifier[active_mask]
        active_imgs_tensor = imgs_tensor[active_mask]
        active_labs_tensor = labs_tensor[active_mask]
        active_CONST = CONST[active_mask]

        optimizer.zero_grad()

        modified_input = active_imgs_tensor + active_modifier
        newimg = sin_func_torch(modified_input)
        output = model(newimg)

        l2dist = torch.sum((newimg - imgs[active_mask]) ** 2, dim=1)

        real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
        other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

        loss1 = torch.clamp((other - real + confidence), min=0.0)
        loss2 = l2dist

        if outer_step != (binary_search_steps - 1):
            loss = (active_CONST * loss1) + loss2
        else:
            loss = loss1

        total_loss = torch.sum(loss)
        total_loss.backward()
        optimizer.step()
        scheduler.step()

        with torch.no_grad():
            improved_mask = loss < best_loss[active_mask]
            best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
            no_improvement_count[active_mask] = torch.where(improved_mask,
                torch.zeros_like(no_improvement_count[active_mask]),
                no_improvement_count[active_mask] + 1)

            improved_mask2 = (loss1 < o_best_loss1[active_mask])
            o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
            o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

            update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
            bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
            bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

            update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
            o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
            o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
            o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

            # Skip the rest of the loop for samples that haven't improved in 50 iterations
            #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

    if outer_step != (binary_search_steps - 1):
        success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
        o_success_attack  = torch.logical_or(o_success_attack, success_mask)
        print('o_success_attack ', o_success_attack.sum())

        failure_mask = ~success_mask
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

success_attack_mask = (o_bestl2 != 1e10)
final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

In [ ]:
def final_rounding(oimgs, newimg, model):
    # Ensure all tensors are on the GPU
    oimgs = oimgs.to(device)
    perturbed = newimg.to(device)

    # Calculate the absolute difference between original and perturbed images
    total_change = (perturbed - oimgs).abs()

    # Create a mask for elements with significant changes
    significant_mask = total_change > 0.1

    # Prepare a copy of the original images to apply the final rounding
    final = oimgs.clone()

    # Calculate the direction of changes: 0 for decrease/no change, 1 for increase
    changes = torch.where(perturbed > oimgs, torch.tensor(1.0, device=device), torch.tensor(0.0, device=device))

    # Set the model to evaluation mode
    model.eval()

    # Iterate over each image in the batch
    for b in range(total_change.shape[0]):
        # Get indices of significant changes, sorted by magnitude
        sorted_idx = torch.argsort(total_change[b][significant_mask[b]], descending=True)

        # Map sorted indices back to the original image indices
        relevant_indices = torch.nonzero(significant_mask[b], as_tuple=True)[0][sorted_idx]

        # Apply changes in order of significance
        for idx in relevant_indices:
            # Update the final image tensor with the change
            final[b, idx] = changes[b, idx]

            # Check the model's prediction; stop if the desired outcome is achieved
            if model(final[b].unsqueeze(0)).argmax() == 0:
                break

    return final

In [ ]:
def final_rounding2(oimgs, newimg, model):
    # Ensure all tensors are on the GPU
    oimgs = oimgs.to(device)
    perturbed = newimg.to(device)

    # Set the model to evaluation mode
    model.eval()

    # Initialize active_mask based on model's initial prediction
    active_mask = (model(oimgs).argmax(dim=1) == 1)

    # Calculate the absolute difference between original and perturbed images
    total_change = (perturbed - oimgs).abs()

    # Determine which changes are significant
    significant_mask = total_change > 0.1

    # Prepare a copy of the original images to apply the final rounding
    final = oimgs.clone()

    # Determine the direction of changes: 0 for decrease/no change, 1 for increase
    changes = (perturbed > oimgs).float()

    # Get sorted indices of the total changes for each image in descending order
    sorted_idx = total_change.argsort(dim=1, descending=True)

    # Get the maximum number of significant changes across all images
    max_significant_changes = significant_mask.sum(dim=1).max()

    # Iterate over each significant change index
    for i in range(max_significant_changes):
        # Update active_mask indices
        active_indices = torch.nonzero(active_mask).squeeze()

        if active_indices.numel() == 0:
            break

        # Safeguard when active_indices is a single-element tensor
        if active_indices.dim() == 0:
            active_indices = active_indices.unsqueeze(0)

        # Get the relevant indices for current change
        indices = sorted_idx[active_indices, i]

        # Apply the changes to the final image tensor
        final[active_indices, indices] = changes[active_indices, indices]

        # Update the active_mask based on the model's output
        active_mask[active_indices] = (model(final[active_indices]).argmax(dim=1) == 1)

    return final

In [ ]:
final = final_rounding(imgs, final_output, model)
mask = (model(final).argmax(1) == 0)
print(((final[mask] - imgs[mask]).abs()).sum())
print((mask))

### Step 17 — Attack on the combined model, scored three ways

# all data

In [ ]:
def final_rounding(oimgs, newimg, model):
    # Ensure all tensors are on the GPU
    oimgs = oimgs.to(device)
    perturbed = newimg.to(device)

    # Calculate the absolute difference between original and perturbed images
    total_change = (perturbed - oimgs).abs()

    # Create a mask for elements with significant changes
    significant_mask = total_change > 0.05

    # Prepare a copy of the original images to apply the final rounding
    final = oimgs.clone()

    # Calculate the direction of changes: 0 for decrease/no change, 1 for increase
    changes = torch.where(perturbed > oimgs, torch.tensor(1.0, device=device), torch.tensor(0.0, device=device))

    # Set the model to evaluation mode
    model.eval()

    # Iterate over each image in the batch
    for b in range(total_change.shape[0]):
        # Get indices of significant changes, sorted by magnitude
        sorted_idx = torch.argsort(total_change[b][significant_mask[b]], descending=True)

        # Map sorted indices back to the original image indices
        relevant_indices = torch.nonzero(significant_mask[b], as_tuple=True)[0][sorted_idx]

        # Apply changes in order of significance
        for idx in relevant_indices:
            # Update the final image tensor with the change
            final[b, idx] = changes[b, idx]

            # Check the model's prediction; stop if the desired outcome is achieved
            if model(final[b].unsqueeze(0)).argmax() == 0:
                break

    return final

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= 0.001  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += 0.001  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L2 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=0.1, step_size_up=4,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if (iteration + 1) % 2000 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Iteration {iteration + 1}: Current Learning Rate: {current_lr}")

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs[active_mask]) ** 2, dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + loss2
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask,
                    torch.zeros_like(no_improvement_count[active_mask]),
                    no_improvement_count[active_mask] + 1)

                improved_mask2 = (loss1 < o_best_loss1[active_mask])
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1):
            success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
            o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            print('o_success_attack ', o_success_attack.sum())

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

**`Attack on combined_model`**

In [ ]:
#CW(mal_x_batch, mal_y_batch, model_AT_rFGSM, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, device=device)
attack_params =  {'device':device, 'confidence':0.1, 'removal_array':removal_array,'max_iterations': 20000, 'learning_rate': 0.3, 'initial_const': 0.1 , 'binary_search_steps' : 11, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, combined_model, CW, **attack_params)

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]

In [ ]:
# Evaluation on non-rounded adversarial examples

# Set the model to evaluation mode
combined_model.eval()

# Get the model's predictions on adversarial examples
outputs = combined_model(adv)
y_pred = outputs.argmax(dim=1)

# Calculate metrics
num_malwares = len(mal_y_batch)
num_successful_attacks = (y_pred == 0).sum().item()
total_difference = (torch.abs(adv[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
accuracy = (y_pred == 1).sum().item() / num_malwares * 100

# Print results with clear formatting
print(f"Evaluation Results on NOT_Rounded Adversarial Examples")
print(f"-----------------------------------------------")
print(f"Number of malware samples: {num_malwares}")
print(f"Number of successful attacks: {num_successful_attacks}")
print(f"Total difference (L1 norm) between successful adversarial examples and original: {total_difference}")
print(f"Accuracy of the model on malwares under attack: {accuracy:.2f}%")

In [ ]:
# Evaluation on rounded adversarial examples(naive technique)

# Set the model to evaluation mode
combined_model.eval()

# Apply the final rounding solution to adversarial examples
adv_rounded = round_x(adv, round_threshold=0.5)

# Get the model's predictions on the rounded adversarial examples
outputs = combined_model(adv_rounded)
y_pred = outputs.argmax(dim=1)

# Calculate evaluation metrics
num_malwares = len(mal_y_batch)
num_successful_attacks = (y_pred == 0).sum().item()
total_difference = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
accuracy = (y_pred == 1).sum().item() / num_malwares * 100

# Display the results with clear formatting
print(f"Evaluation Results on Rounded Adversarial Examples")
print(f"-----------------------------------------------")
print(f"Number of malware samples: {num_malwares}")
print(f"Number of successful attacks: {num_successful_attacks}")
print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

In [ ]:
# Evaluation on rounded adversarial examples

# Set the model to evaluation mode
combined_model.eval()

# Apply the final rounding solution to adversarial examples
adv_rounded = final_rounding(mal_x_batch, adv, combined_model)

# Get the model's predictions on the rounded adversarial examples
outputs = combined_model(adv_rounded)
y_pred = outputs.argmax(dim=1)

# Calculate evaluation metrics
num_malwares = len(mal_y_batch)
num_successful_attacks = (y_pred == 0).sum().item()
total_difference = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
accuracy = (y_pred == 1).sum().item() / num_malwares * 100

# Display the results with clear formatting
print(f"Evaluation Results on Rounded Adversarial Examples")
print(f"-----------------------------------------------")
print(f"Number of malware samples: {num_malwares}")
print(f"Number of successful attacks: {num_successful_attacks}")
print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

### Step 18 — Attack on the adversarially trained model

**`Attack on model_AT_rFGSM`**

In [ ]:
#CW(mal_x_batch, mal_y_batch, model_AT_rFGSM, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, device=device)
attack_params =  {'device':device, 'confidence':0.1, 'removal_array':removal_array,'max_iterations': 20000, 'learning_rate': 0.3, 'initial_const': 0.01 , 'binary_search_steps' : 11, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, model_AT_rFGSM, CW, **attack_params)

In [ ]:
# Evaluation on non-rounded adversarial examples

# Set the model to evaluation mode
model_AT_rFGSM.eval()

# Get the model's predictions on adversarial examples
outputs = model_AT_rFGSM(adv)
y_pred = outputs.argmax(dim=1)

# Calculate metrics
num_malwares = len(mal_y_batch)
num_successful_attacks = (y_pred == 0).sum().item()
total_difference = (torch.abs(adv[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
accuracy = (y_pred == 1).sum().item() / num_malwares * 100

# Print results with clear formatting
print(f"Evaluation Results on NOT_Rounded Adversarial Examples")
print(f"-----------------------------------------------")
print(f"Number of malware samples: {num_malwares}")
print(f"Number of successful attacks: {num_successful_attacks}")
print(f"Total difference (L1 norm) between successful adversarial examples and original: {total_difference}")
print(f"Accuracy of the model on malwares under attack: {accuracy:.2f}%")

In [ ]:
# Evaluation on rounded adversarial examples(naive technique)

# Set the model to evaluation mode
model_AT_rFGSM.eval()

# Apply the final rounding solution to adversarial examples
adv_rounded = round_x(adv, round_threshold=0.5)

# Get the model's predictions on the rounded adversarial examples
outputs = model_AT_rFGSM(adv_rounded)
y_pred = outputs.argmax(dim=1)

# Calculate evaluation metrics
num_malwares = len(mal_y_batch)
num_successful_attacks = (y_pred == 0).sum().item()
total_difference = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
accuracy = (y_pred == 1).sum().item() / num_malwares * 100

# Display the results with clear formatting
print(f"Evaluation Results on Rounded Adversarial Examples")
print(f"-----------------------------------------------")
print(f"Number of malware samples: {num_malwares}")
print(f"Number of successful attacks: {num_successful_attacks}")
print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

In [ ]:
# Evaluation on rounded adversarial examples

# Set the model to evaluation mode
model_AT_rFGSM.eval()

# Apply the final rounding solution to adversarial examples
adv_rounded = final_rounding(mal_x_batch, adv, model_AT_rFGSM)

# Get the model's predictions on the rounded adversarial examples
outputs = model_AT_rFGSM(adv_rounded)
y_pred = outputs.argmax(dim=1)

# Calculate evaluation metrics
num_malwares = len(mal_y_batch)
num_successful_attacks = (y_pred == 0).sum().item()
total_difference = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
accuracy = (y_pred == 1).sum().item() / num_malwares * 100

# Display the results with clear formatting
print(f"Evaluation Results on Rounded Adversarial Examples")
print(f"-----------------------------------------------")
print(f"Number of malware samples: {num_malwares}")
print(f"Number of successful attacks: {num_successful_attacks}")
print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

In [ ]:
#loss = (real - other)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10000, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv = adv_predict(test_loader, model_AT_rFGSM, pgd_min22, device, **attack_params)

In [ ]:
# Evaluation on rounded adversarial examples

# Set the model to evaluation mode
model_AT_rFGSM.eval()

# Apply the final rounding solution to adversarial examples
adv_rounded2 = final_rounding(mal_x_batch, adv, model_AT_rFGSM)

# Get the model's predictions on the rounded adversarial examples
outputs = model_AT_rFGSM(adv_rounded2)
y_pred = outputs.argmax(dim=1)

# Calculate evaluation metrics
num_malwares = len(mal_y_batch)
num_successful_attacks = (y_pred == 0).sum().item()
total_difference = (torch.abs(adv_rounded2[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
accuracy = (y_pred == 1).sum().item() / num_malwares * 100

# Display the results with clear formatting
print(f"Evaluation Results on Rounded Adversarial Examples")
print(f"-----------------------------------------------")
print(f"Number of malware samples: {num_malwares}")
print(f"Number of successful attacks: {num_successful_attacks}")
print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

### Step 19 — Loss variants: the quarter-circle loss

# loss function (a quarter of circle):


```
(1 - (perturbation_tensor - 1)**2)**0.5
```



In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += confidence  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L2 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=0.1, step_size_up=4,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        iteration = 0
        while iteration < max_iterations:

            if active_mask.sum() == 0:
                print(f"All samples are inactive at iteration {iteration}.")
                best_loss = torch.full((batch_size,), float('inf'), device=device)
                no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
                active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)
                if iteration < 5000:
                    steps_to_skip = 5000 - iteration
                    iteration = 5000  # Jump to iteration 5000
                elif 5000 < iteration < 10000:
                    steps_to_skip = 10000 - iteration
                    iteration = 10000  # Jump to iteration 10000
                elif 10000 < iteration < 15000:
                    steps_to_skip = 15000 - iteration
                    iteration = 15000  # Jump to iteration 15000
                elif iteration > 15000:
                    break  # Exit the loop

                # Step the scheduler to the correct point in the cycle
                for _ in range(steps_to_skip):
                    scheduler.step()

            else:
                active_indices = torch.where(active_mask)[0]
                active_modifier = modifier[active_mask]
                active_imgs_tensor = imgs_tensor[active_mask]
                active_labs_tensor = labs_tensor[active_mask]
                active_CONST = CONST[active_mask]

                optimizer.zero_grad()

                modified_input = active_imgs_tensor + active_modifier
                newimg = sin_func_torch(modified_input)
                output = model(newimg)

                perturbation = torch.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor))
                eps = 1e-2
                l2dist = torch.sum((1 - (perturbation - 1)**2 + eps)**0.5 - eps**0.5 , dim=1)
                #l2dist = torch.sum((newimg - imgs[active_mask]) ** 2, dim=1)

                real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
                other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

                loss1 = torch.clamp((other - real + confidence), min=0.0)

                if outer_step != (binary_search_steps - 1):
                    loss = (active_CONST * loss1) + l2dist
                else:
                    loss = loss1

                total_loss = torch.sum(loss)
                total_loss.backward()
                optimizer.step()
                scheduler.step()

                with torch.no_grad():
                    improved_mask = loss < best_loss[active_mask]
                    best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                    no_improvement_count[active_mask] = torch.where(improved_mask,
                        torch.zeros_like(no_improvement_count[active_mask]),
                        no_improvement_count[active_mask] + 1)

                    improved_mask2 = (loss1 < o_best_loss1[active_mask])
                    o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                    o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                    success_mask_inloop = compare(output, active_labs_tensor, confidence, targeted)

                    update_mask = (l2dist < bestl2[active_mask]) & success_mask_inloop
                    bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                    bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                    update_mask2 = (l2dist < o_bestl2[active_mask]) & success_mask_inloop
                    o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                    o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                    o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                    if (iteration + 1) % 500 == 0:
                        success_mask[active_mask]  = torch.logical_or(success_mask_inloop, success_mask[active_mask])
                        current_lr = scheduler.get_last_lr()[0]
                        print(f"Iteration {iteration + 1}: Current success : {success_mask_inloop.sum()}, All success : {success_mask.sum()}, mean(l2) : {o_bestl2[success_mask].mean():.6f}, Current Learning Rate: {current_lr:.2f}")

                        #rint(f"Iteration {iteration + 1}: Current success : {success_mask_inloop.sum()}, All success : {success_mask.sum()}, Current Learning Rate: {current_lr}")


                    # Skip the rest of the loop for samples that haven't improved in 50 iterations
                    active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100

                    iteration += 1  # Increment the iteration manually

        if outer_step != (binary_search_steps - 1):
            o_success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
            o_success_attack  = torch.logical_or(o_success_attack, o_success_mask)
            print('o_success_attack ', o_success_attack.sum())
            print('-------------------------------------------------')

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += confidence  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L2 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=0.1, step_size_up=5,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            perturbation = torch.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor))
            eps = 1e-2
            l2dist = torch.sum((1 - (perturbation - 1)**2 + eps)**0.5 - eps**0.5 , dim=1)
            #l2dist = torch.sum((newimg - imgs[active_mask]) ** 2, dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + l2dist
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask,
                    torch.zeros_like(no_improvement_count[active_mask]),
                    no_improvement_count[active_mask] + 1)

                improved_mask2 = (loss1 < o_best_loss1[active_mask])
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                success_mask_inloop = compare(output, active_labs_tensor, confidence, targeted)

                update_mask = (l2dist < bestl2[active_mask]) & success_mask_inloop
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & success_mask_inloop
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                if (iteration + 1) % 500 == 0:
                    success_mask  = torch.logical_or(success_mask_inloop, success_mask)
                    current_lr = scheduler.get_last_lr()[0]
                    print(f"Iteration {iteration + 1}: Current success : {success_mask_inloop.sum()} \t All success : {success_mask.sum()} \t mean(l2) : {o_bestl2[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")
                    #print(f"Iteration {iteration + 1}: Current success : {success_mask_inloop.sum()}, All success : {success_mask.sum()}, Current Learning Rate: {current_lr}")


                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1):
            o_success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
            o_success_attack  = torch.logical_or(o_success_attack, o_success_mask)
            print('o_success_attack ', o_success_attack.sum())
            print('-------------------------------------------------')

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
#CW(mal_x_batch, mal_y_batch, model_AT_rFGSM, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, device=device)
attack_params =  {'device':device, 'confidence':0.1, 'removal_array':removal_array,'max_iterations': 20000, 'learning_rate': 0.3, 'initial_const': 1 , 'binary_search_steps' : 8, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, combined_model, CW, **attack_params)

### Step 20 — Loss variants: L1, and the L0 penalty

# L1 NORM

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= 0.001  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += 0.001  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=0.1, step_size_up=4,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if (iteration + 1) % 2000 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Iteration {iteration + 1}: Current Learning Rate: {current_lr}")

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            l2dist = torch.sum(torch.abs(newimg - imgs[active_mask]), dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + loss2
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask,
                    torch.zeros_like(no_improvement_count[active_mask]),
                    no_improvement_count[active_mask] + 1)

                improved_mask2 = (loss1 < o_best_loss1[active_mask])
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1):
            success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
            o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            print('o_success_attack ', o_success_attack.sum())

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
#CW(mal_x_batch, mal_y_batch, model_AT_rFGSM, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, device=device)
attack_params =  {'device':device, 'confidence':0.1, 'removal_array':removal_array,'max_iterations': 20000, 'learning_rate': 0.3, 'initial_const': 0.1 , 'binary_search_steps' : 11, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, combined_model, CW_l1, **attack_params)

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= 0.001  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += 0.001  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=0.1, step_size_up=4,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if (iteration + 1) % 2000 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Iteration {iteration + 1}: Current Learning Rate: {current_lr}")

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            epsilon = 1e-1
            perturbation = torch.abs(active_modifier)
            l2dist = torch.sum(perturbation / (perturbation + epsilon), dim=1)
            #torch.sum(torch.abs(newimg - imgs[active_mask]), dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + loss2
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask,
                    torch.zeros_like(no_improvement_count[active_mask]),
                    no_improvement_count[active_mask] + 1)

                improved_mask2 = (loss1 < o_best_loss1[active_mask])
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1):
            success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
            o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            print('o_success_attack ', o_success_attack.sum())

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e9
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
def l0_penalty(original_data, adv_data):
    perturbation = original_data - adv_data
    return torch.sum(perturbation != 0).float()

l0_penalty(mal_x_batch[y_pred == 0], adv[y_pred == 0])

### Step 21 — Loss variants: the custom sparsity losses

# custom function for loss



```
sparsity_loss = perturbation**2 / (perturbation**2 + epsilon**2)
```



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# Define parameters
epsilon = 0.001
perturbation = np.linspace(-1, 1, 10001)

# Calculate sparsity loss
sparsity_loss = perturbation**2 / (perturbation**2 + epsilon)

# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(perturbation, sparsity_loss, label='Sparsity-Inducing Loss')
plt.title('Sparsity-Inducing Loss vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity-Inducing Loss')
plt.legend()
plt.grid(True)
plt.show()

# Convert perturbation to a PyTorch tensor
perturbation_tensor = torch.tensor(perturbation, requires_grad=True)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = perturbation_tensor**2 / (perturbation_tensor**2 + epsilon)

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = perturbation_tensor.grad

# Plot the gradient
plt.figure(figsize=(12, 6))
plt.plot(perturbation, gradient.detach().numpy(), label='Numerical Derivative (Gradient)')
plt.title('Numerical Derivative of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= 0.001  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += 0.001  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=0.1, step_size_up=4,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if (iteration + 1) % 2000 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Iteration {iteration + 1}: Current Learning Rate: {current_lr}")

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            epsilon = torch.tensor(0.001, device=newimg.device, dtype=newimg.dtype)
            perturbation = (newimg - imgs[active_mask]).square()
            l2dist = torch.sum(perturbation / (perturbation + epsilon), dim=1)
            #torch.sum(perturbation / (perturbation + epsilon), dim=1)
            #torch.sum(torch.abs(newimg - imgs[active_mask]), dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            #loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + (l2dist / 20.)
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask,
                    torch.zeros_like(no_improvement_count[active_mask]),
                    no_improvement_count[active_mask] + 1)

                improved_mask2 = (loss1 < o_best_loss1[active_mask])
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1):
            success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
            o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            print('o_success_attack ', o_success_attack.sum())

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e10
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

# custom function for loss2






```
sparsity_loss = perturbation**2 / (perturbation**2 + epsilon**2)
```



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# Define parameters
epsilon = 0.0001
perturbation = np.linspace(-1, 1, 10001)

# Calculate sparsity loss
sparsity_loss = perturbation**2 / (perturbation**2 + epsilon)

# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(perturbation, sparsity_loss, label='Sparsity-Inducing Loss')
plt.title('Sparsity-Inducing Loss vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity-Inducing Loss')
plt.legend()
plt.grid(True)
plt.show()

# Convert perturbation to a PyTorch tensor
perturbation_tensor = torch.tensor(perturbation, requires_grad=True)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = perturbation_tensor**2 / (perturbation_tensor**2 + epsilon)

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = perturbation_tensor.grad

# Plot the gradient
plt.figure(figsize=(12, 6))
plt.plot(perturbation, gradient.detach().numpy(), label='Numerical Derivative (Gradient)')
plt.title('Numerical Derivative of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= 0.001  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += 0.001  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.00001, max_lr=0.1, step_size_up=5,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if (iteration + 1) % 2000 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Iteration {iteration + 1}: Current Learning Rate: {current_lr}")

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            epsilon = torch.tensor(0.0001, device=newimg.device, dtype=newimg.dtype)
            perturbation = (newimg - imgs[active_mask]).square()
            l2dist = torch.sum(perturbation / (perturbation + epsilon), dim=1)
            #torch.sum(perturbation / (perturbation + epsilon), dim=1)
            #torch.sum(torch.abs(newimg - imgs[active_mask]), dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            #loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + (l2dist / 20.)
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask,
                    torch.zeros_like(no_improvement_count[active_mask]),
                    no_improvement_count[active_mask] + 1)

                improved_mask2 = (loss1 < o_best_loss1[active_mask])
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1):
            success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
            o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            print('o_success_attack ', o_success_attack.sum())

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e10
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

### Step 22 — Loss variants: new rounding functions

# New function

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= 0.001  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += 0.001  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.00001, max_lr=0.1, step_size_up=5,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if (iteration + 1) % 2000 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Iteration {iteration + 1}: Current Learning Rate: {current_lr}")

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            #perturbation = torch.abs(newimg - imgs[active_mask])
            #l2dist = torch.sum(perturbation , dim=1)

            l2dist = torch.sum(torch.abs(newimg - imgs[active_mask]), dim=1)
            #torch.sum(perturbation / (perturbation + epsilon), dim=1)
            #torch.sum(torch.abs(newimg - imgs[active_mask]), dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            #loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + l2dist
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask,
                    torch.zeros_like(no_improvement_count[active_mask]),
                    no_improvement_count[active_mask] + 1)

                improved_mask2 = (loss1 < o_best_loss1[active_mask])
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1):
            success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
            o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            print('o_success_attack ', o_success_attack.sum())

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e10
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
#CW(mal_x_batch, mal_y_batch, model_AT_rFGSM, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, device=device)
attack_params =  {'device':device, 'confidence':0.1, 'removal_array':removal_array,'max_iterations': 20000, 'learning_rate': 0.3, 'initial_const': 0.1 , 'binary_search_steps' : 9, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, combined_model, CW_l1, **attack_params)

# New function2

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence, targeted):
    """
    Compares the output with the target for a targeted or untargeted attack.

    For targeted attacks, it checks if the output class matches the target class.
    For untargeted attacks, it checks if the output class does not match the target class.
    """
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        else:
            output[target] += confidence  # Slightly increase the confidence for untargeted attacks
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def compare_round(mal_x_batch,newimg, model):

    adv_rounded = final_rounding(mal_x_batch, newimg, model)
    logits = model(adv_rounded)

    output = torch.argmax(logits, dim=1)
    return output == 0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]
    targeted = True

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.00001, max_lr=0.1, step_size_up=5,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if (iteration + 1) % 2000 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Iteration {iteration + 1}: Current Learning Rate: {current_lr}")

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            perturbation = torch.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor))

            l2dist = torch.sum(perturbation, dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + l2dist
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask,
                    torch.zeros_like(no_improvement_count[active_mask]),
                    no_improvement_count[active_mask] + 1)

                improved_mask2 = (loss1 < o_best_loss1[active_mask])
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1):
            success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
            o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            print('o_success_attack ', o_success_attack.sum())

            o_success_attack_round = compare_round(imgs_tensor,o_bestattack, model)
            print('o_success_attack_round ', o_success_attack_round.sum())
            print('----------------------------------------')

            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e10
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
#CW(mal_x_batch, mal_y_batch, model_AT_rFGSM, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, device=device)
attack_params =  {'device':device, 'confidence':0.5, 'removal_array':removal_array,'max_iterations': 20000, 'learning_rate': 0.3, 'initial_const': 0.1 , 'binary_search_steps' : 9, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, combined_model, CW_l1, **attack_params)

### Step 23 — Rounding the whole tensor at once, and the confidence sweep

# New approach
- rounding whole input tensor at once

- new function for loss

- our criterion is rounded adversarial examples during training

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            adv_rounded = final_rounding(mal_x_batch, pertb_mal_x, model,removal_array)

            outputs = model(adv_rounded)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            #print(outputs)
            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            #print((y_pred == 0))
            #print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            #print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return pertb_mal_x

In [ ]:
def final_rounding(oimgs, newimg, model, removal_array):
    # Expand removal array to match the batch size
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8))

    # Ensure all tensors are on the correct device
    oimgs = oimgs.to(device)
    perturbed = newimg.to(device)

    # Set the model to evaluation mode
    model.eval()

    # Initialize active_mask based on model's initial prediction
    active_mask = (model(oimgs).argmax(dim=1) == 1)

    # Calculate the absolute difference and direction of changes
    total_change = (perturbed - oimgs).abs()
    changes = (perturbed > oimgs).float()

    # Determine significant changes
    significant_mask = total_change > 0.1

    # Sort indices of changes in descending order of significance
    sorted_idx = total_change.argsort(dim=1, descending=True)

    # Copy original images to apply final rounding
    final = oimgs.clone()

    # Iterate up to the maximum number of significant changes
    max_significant_changes = significant_mask.sum(dim=1).max().item()

    for i in range(max_significant_changes):
        # Safeguard against empty active mask
        if not active_mask.any():
            break

        # Get the indices of active samples
        active_indices = torch.where(active_mask)[0]

        # Get the relevant indices for current change
        indices = sorted_idx[active_indices, i]

        # since max_significant_changes is relatively large for some samples, possible that zero changes which belong to fixed_features get updated
        final[active_indices, indices] = torch.where(non_fixed_features_mask[active_indices, indices],
                                                     changes[active_indices, indices], final[active_indices, indices])

        # Update the active_mask based on the model's prediction
        active_mask[active_indices] = (model(final[active_indices]).argmax(dim=1) == 1)

    return final

#(1 - (perturbation - 1)^2 + eps)^0.5 - eps^0.5

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence):
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        output = torch.argmax(output, dim=1)
    return output == target

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, max_learning_rate=0.1,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl0 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=max_learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=max_learning_rate, step_size_up=5,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl0 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            eps = 1e-2
            perturbation = torch.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor))

            l2dist = torch.sum((1 - (perturbation - 1)**2 + eps)**0.5 - eps**0.5 , dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            #loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + l2dist
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            if (iteration + 1) % 10 == 0:
                with torch.no_grad():
                    success_mask_inloop, l0dist = compare_round(active_imgs_tensor,newimg, model,removal_array)
                    success_mask  = torch.logical_or(success_mask_inloop, success_mask)
                    current_lr = scheduler.get_last_lr()[0]

                    improved_mask2 = (loss1 < o_best_loss1[active_mask])
                    o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                    o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                    update_mask = (l0dist < bestl0[active_mask]) & success_mask_inloop
                    bestl0[active_mask] = torch.where(update_mask, l0dist, bestl0[active_mask])
                    bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                    update_mask2 = (l0dist < o_bestl0[active_mask]) & success_mask_inloop
                    o_bestl0[active_mask] = torch.where(update_mask2, l0dist, o_bestl0[active_mask])
                    o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                    # Skip the rest of the loop for samples that haven't improved in 50 iterations
                    #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

                    if (iteration + 1) % 100 == 0:
                        print(f"Iteration {iteration + 1}: Current success : {success_mask_inloop.sum()} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl0[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")

        if outer_step != (binary_search_steps - 1):
            #o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            o_success_attack, o_l0dist = compare_round(imgs,o_bestattack, model,removal_array)
            print(f"outer_step {outer_step + 1}: all success : {o_success_attack.sum()} \t mean(l0)(success) : {o_l0dist[o_success_attack].mean():.4f}")
            #print('o_success_attack ', o_success_attack.sum())
            print('-------------------------------------------------------------')
            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e10
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl0 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
# eps=1e4
attack_params =  {'device':device, 'confidence':0.5, 'removal_array':removal_array,'max_iterations': 20000, 'max_learning_rate': 0.1, 'initial_const': 1. , 'binary_search_steps' : 5, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, model_AT_rFGSM, CW_l1, **attack_params)

In [ ]:
# Evaluation on rounded adversarial examples

# Set the model to evaluation mode
model_AT_rFGSM.eval()

# Apply the final rounding solution to adversarial examples
adv_rounded = final_rounding(mal_x_batch, adv, model_AT_rFGSM,removal_array)

# Get the model's predictions on the rounded adversarial examples
outputs = model_AT_rFGSM(adv_rounded)
y_pred = outputs.argmax(dim=1)

# Calculate evaluation metrics
num_malwares = len(mal_y_batch)
num_successful_attacks = (y_pred == 0).sum().item()
total_difference = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
accuracy = (y_pred == 1).sum().item() / num_malwares * 100

# Display the results with clear formatting
print(f"Evaluation Results on Rounded Adversarial Examples")
print(f"-----------------------------------------------")
print(f"Number of malware samples: {num_malwares}")
print(f"Number of successful attacks: {num_successful_attacks}")
print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming differences is already computed as in your code
differences = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum(1)

# Convert differences to a numpy array for plotting
differences_np = differences.cpu().numpy()

# Set the style of the plot
sns.set(style="whitegrid")

# Create a histogram with Seaborn
plt.figure(figsize=(10, 6))
sns.histplot(differences_np, bins=16, kde=False, color='skyblue')
#sns.histplot(differences_np, bins=30, kde=True, color='skyblue')

# Set the title and labels
plt.title('Abundance of Differences', fontsize=16)
plt.xlabel('Sum of Absolute Differences', fontsize=14)
plt.ylabel('Frequency', fontsize=14)

# Show the plot
plt.show()

In [ ]:
attack_params =  {'device':device, 'confidence':0.5, 'removal_array':removal_array,'max_iterations': 20000, 'max_learning_rate': 0.1, 'initial_const': 1 , 'binary_search_steps' : 7, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, combined_model, CW_l1, **attack_params)

In [ ]:
# Evaluation on rounded adversarial examples

# Set the model to evaluation mode
combined_model.eval()

# Apply the final rounding solution to adversarial examples
adv_rounded = final_rounding(mal_x_batch, adv, combined_model,removal_array)

# Get the model's predictions on the rounded adversarial examples
outputs = combined_model(adv_rounded)
y_pred = outputs.argmax(dim=1)

# Calculate evaluation metrics
num_malwares = len(mal_y_batch)
num_successful_attacks = (y_pred == 0).sum().item()
total_difference = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
accuracy = (y_pred == 1).sum().item() / num_malwares * 100

# Display the results with clear formatting
print(f"Evaluation Results on Rounded Adversarial Examples")
print(f"-----------------------------------------------")
print(f"Number of malware samples: {num_malwares}")
print(f"Number of successful attacks: {num_successful_attacks}")
print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

### Step 24 — Loss variants: perturbation², perturbation, and the rational penalty

# pertubation ** 2

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence):
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        output = torch.argmax(output, dim=1)
    return output == target

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, max_learning_rate=0.1,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl0 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=max_learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=max_learning_rate, step_size_up=5,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl0 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            #eps = 1e-2
            #perturbation = torch.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor))

            #l2dist = torch.sum((1 - (perturbation - 1)**2 + eps)**0.5 - eps**0.5 , dim=1)
            l2dist = torch.sum((newimg - active_imgs_tensor) ** 2, dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            #loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + l2dist
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            if (iteration + 1) % 10 == 0:
                with torch.no_grad():
                    success_mask_inloop, l0dist = compare_round(active_imgs_tensor,newimg, model,removal_array)
                    success_mask  = torch.logical_or(success_mask_inloop, success_mask)
                    current_lr = scheduler.get_last_lr()[0]

                    improved_mask2 = (loss1 < o_best_loss1[active_mask])
                    o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                    o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                    update_mask = (l0dist < bestl0[active_mask]) & success_mask_inloop
                    bestl0[active_mask] = torch.where(update_mask, l0dist, bestl0[active_mask])
                    bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                    update_mask2 = (l0dist < o_bestl0[active_mask]) & success_mask_inloop
                    o_bestl0[active_mask] = torch.where(update_mask2, l0dist, o_bestl0[active_mask])
                    o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                    # Skip the rest of the loop for samples that haven't improved in 50 iterations
                    #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

                    if (iteration + 1) % 100 == 0:
                        print(f"Iteration {iteration + 1}: Current success : {success_mask_inloop.sum()} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl0[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")

        if outer_step != (binary_search_steps - 1):
            #o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            o_success_attack, o_l0dist = compare_round(imgs,o_bestattack, model,removal_array)
            print(f"outer_step {outer_step + 1}: all success : {o_success_attack.sum()} \t mean(l0)(success) : {o_l0dist[o_success_attack].mean():.4f}")
            #print('o_success_attack ', o_success_attack.sum())
            print('-------------------------------------------------------------')
            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e10
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl0 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
#l2dist = torch.sum(perturbation ** 2, dim=1)
attack_params =  {'device':device, 'confidence':0.5, 'removal_array':removal_array,'max_iterations': 20000, 'max_learning_rate': 0.1, 'initial_const': 1 , 'binary_search_steps' : 7, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, combined_model, CW_l1, **attack_params)

#perturbation

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence):
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        output = torch.argmax(output, dim=1)
    return output == target

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, max_learning_rate=0.1,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl0 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=max_learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=max_learning_rate, step_size_up=5,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl0 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            #eps = 1e-2
            perturbation = torch.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor))

            #l2dist = torch.sum((1 - (perturbation - 1)**2 + eps)**0.5 - eps**0.5 , dim=1)
            #l2dist = torch.sum((newimg - active_imgs_tensor) ** 2, dim=1)
            l2dist = torch.sum(perturbation, dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            #loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + l2dist
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            if (iteration + 1) % 10 == 0:
                with torch.no_grad():
                    success_mask_inloop, l0dist = compare_round(active_imgs_tensor,newimg, model,removal_array)
                    success_mask  = torch.logical_or(success_mask_inloop, success_mask)
                    current_lr = scheduler.get_last_lr()[0]

                    improved_mask2 = (loss1 < o_best_loss1[active_mask])
                    o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                    o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                    update_mask = (l0dist < bestl0[active_mask]) & success_mask_inloop
                    bestl0[active_mask] = torch.where(update_mask, l0dist, bestl0[active_mask])
                    bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                    update_mask2 = (l0dist < o_bestl0[active_mask]) & success_mask_inloop
                    o_bestl0[active_mask] = torch.where(update_mask2, l0dist, o_bestl0[active_mask])
                    o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                    # Skip the rest of the loop for samples that haven't improved in 50 iterations
                    #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

                    if (iteration + 1) % 100 == 0:
                        print(f"Iteration {iteration + 1}: Current success : {success_mask_inloop.sum()} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl0[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")

        if outer_step != (binary_search_steps - 1):
            #o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            o_success_attack, o_l0dist = compare_round(imgs,o_bestattack, model,removal_array)
            print(f"outer_step {outer_step + 1}: all success : {o_success_attack.sum()} \t mean(l0)(success) : {o_l0dist[o_success_attack].mean():.4f}")
            #print('o_success_attack ', o_success_attack.sum())
            print('-------------------------------------------------------------')
            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e10
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl0 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
#l2dist = torch.sum(perturbation, dim=1)
attack_params =  {'device':device, 'confidence':0.5, 'removal_array':removal_array,'max_iterations': 20000, 'max_learning_rate': 0.1, 'initial_const': 1 , 'binary_search_steps' : 7, 'cons_increase_factor' : 10}
adv = adv_predict(test_loader, combined_model, CW_l1, **attack_params)

#perturbation^2 / (perturbation^2 + epsilon)

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence):
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        output = torch.argmax(output, dim=1)
    return output == target

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, max_learning_rate=0.1,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl0 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=max_learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=max_learning_rate, step_size_up=5,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl0 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            eps = 0.01
            perturbation_2 = (active_imgs_tensor - newimg)**2

            #l2dist = torch.sum((1 - (perturbation - 1)**2 + eps)**0.5 - eps**0.5 , dim=1)
            #l2dist = torch.sum((newimg - active_imgs_tensor) ** 2, dim=1)
            l2dist = torch.sum((perturbation_2 / (perturbation_2 + eps)), dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            #loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + l2dist
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            if (iteration + 1) % 10 == 0:
                with torch.no_grad():
                    success_mask_inloop, l0dist = compare_round(active_imgs_tensor,newimg, model,removal_array)
                    success_mask  = torch.logical_or(success_mask_inloop, success_mask)
                    current_lr = scheduler.get_last_lr()[0]

                    improved_mask2 = (loss1 < o_best_loss1[active_mask])
                    o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                    o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                    update_mask = (l0dist < bestl0[active_mask]) & success_mask_inloop
                    bestl0[active_mask] = torch.where(update_mask, l0dist, bestl0[active_mask])
                    bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                    update_mask2 = (l0dist < o_bestl0[active_mask]) & success_mask_inloop
                    o_bestl0[active_mask] = torch.where(update_mask2, l0dist, o_bestl0[active_mask])
                    o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                    # Skip the rest of the loop for samples that haven't improved in 50 iterations
                    #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

                    if (iteration + 1) % 100 == 0:
                        print(f"Iteration {iteration + 1}: Current success : {success_mask_inloop.sum()} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl0[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")

        if outer_step != (binary_search_steps - 1):
            #o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            o_success_attack, o_l0dist = compare_round(imgs,o_bestattack, model,removal_array)
            print(f"outer_step {outer_step + 1}: all success : {o_success_attack.sum()} \t mean(l0)(success) : {o_l0dist[o_success_attack].mean():.4f}")
            #print('o_success_attack ', o_success_attack.sum())
            print('-------------------------------------------------------------')
            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e10
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl0 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

### Step 25 — Loss variants: tanh-scaled perturbations


# tanh(2 * perturbation_tensor)

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence):
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        output = torch.argmax(output, dim=1)
    return output == target

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, max_learning_rate=0.1,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl0 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=max_learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=max_learning_rate, step_size_up=5,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl0 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            #eps = 1e-2
            perturbation = torch.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor))

            #l2dist = torch.sum((1 - (perturbation - 1)**2 + eps)**0.5 - eps**0.5 , dim=1)
            #l2dist = torch.sum((newimg - active_imgs_tensor) ** 2, dim=1)
            l2dist = torch.sum(torch.tanh(2 * perturbation), dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            #loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + l2dist
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            if (iteration + 1) % 10 == 0:
                with torch.no_grad():
                    success_mask_inloop, l0dist = compare_round(active_imgs_tensor,newimg, model,removal_array)
                    success_mask  = torch.logical_or(success_mask_inloop, success_mask)
                    current_lr = scheduler.get_last_lr()[0]

                    improved_mask2 = (loss1 < o_best_loss1[active_mask])
                    o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                    o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                    update_mask = (l0dist < bestl0[active_mask]) & success_mask_inloop
                    bestl0[active_mask] = torch.where(update_mask, l0dist, bestl0[active_mask])
                    bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                    update_mask2 = (l0dist < o_bestl0[active_mask]) & success_mask_inloop
                    o_bestl0[active_mask] = torch.where(update_mask2, l0dist, o_bestl0[active_mask])
                    o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                    # Skip the rest of the loop for samples that haven't improved in 50 iterations
                    #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

                    if (iteration + 1) % 100 == 0:
                        print(f"Iteration {iteration + 1}: Current success : {success_mask_inloop.sum()} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl0[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")

        if outer_step != (binary_search_steps - 1):
            #o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            o_success_attack, o_l0dist = compare_round(imgs,o_bestattack, model,removal_array)
            print(f"outer_step {outer_step + 1}: all success : {o_success_attack.sum()} \t mean(l0)(success) : {o_l0dist[o_success_attack].mean():.4f}")
            #print('o_success_attack ', o_success_attack.sum())
            print('-------------------------------------------------------------')
            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e10
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl0 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

# tanh(4 * perturbation_tensor)

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR

def compare(output, target, confidence):
    if not isinstance(output, (float, int, torch.LongTensor, torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        output[target] -= confidence  # Slightly reduce the confidence for targeted attacks
        output = torch.argmax(output, dim=1)
    return output == target

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def CW_l1(imgs, y, model, removal_array, max_iterations=1000, max_learning_rate=0.1,
       binary_search_steps=3, confidence=0.01, initial_const=0.1,
       cons_increase_factor=2, device='cuda'):
    """
    Implements the Carlini & Wagner (CW) L1 attack.

    Parameters:
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - model: The target model to attack.
    - removal_array: A mask array specifying which features are fixed.
    - max_iterations: The maximum number of iterations for the attack.
    - learning_rate: The learning rate for the optimizer.
    - binary_search_steps: The number of steps for the binary search over the constant.
    - confidence: Confidence value for the attack.
    - initial_const: The initial constant for the attack loss.
    - cons_increase_factor: The factor by which the constant is increased during binary search.
    - device: The device on which to perform computations.

    Returns:
    - final_output: The perturbed images after the attack.
    """

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    o_bestl0 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_attack = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    for outer_step in range(binary_search_steps):
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=max_learning_rate)

        def grad_hook(grad):
            return grad * non_fixed_features_mask
        modifier.register_hook(grad_hook)

        scheduler = CyclicLR(optimizer, base_lr=0.0001, max_lr=max_learning_rate, step_size_up=5,
                             step_size_down=4995, mode='triangular2')

        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl0 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(max_iterations):

            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            modified_input = active_imgs_tensor + active_modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            #eps = 1e-2
            perturbation = torch.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor))

            #l2dist = torch.sum((1 - (perturbation - 1)**2 + eps)**0.5 - eps**0.5 , dim=1)
            #l2dist = torch.sum((newimg - active_imgs_tensor) ** 2, dim=1)
            l2dist = torch.sum(torch.tanh(4 * perturbation), dim=1)

            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), active_labs_tensor + 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            #loss2 = l2dist

            if outer_step != (binary_search_steps - 1):
                loss = (active_CONST * loss1) + l2dist
            else:
                loss = loss1

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()
            scheduler.step()

            if (iteration + 1) % 10 == 0:
                with torch.no_grad():
                    success_mask_inloop, l0dist = compare_round(active_imgs_tensor,newimg, model,removal_array)
                    success_mask  = torch.logical_or(success_mask_inloop, success_mask)
                    current_lr = scheduler.get_last_lr()[0]

                    improved_mask2 = (loss1 < o_best_loss1[active_mask])
                    o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                    o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                    update_mask = (l0dist < bestl0[active_mask]) & success_mask_inloop
                    bestl0[active_mask] = torch.where(update_mask, l0dist, bestl0[active_mask])
                    bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                    update_mask2 = (l0dist < o_bestl0[active_mask]) & success_mask_inloop
                    o_bestl0[active_mask] = torch.where(update_mask2, l0dist, o_bestl0[active_mask])
                    o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                    # Skip the rest of the loop for samples that haven't improved in 50 iterations
                    #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

                    if (iteration + 1) % 100 == 0:
                        print(f"Iteration {iteration + 1}: Current success : {success_mask_inloop.sum()} \t All success : {success_mask.sum()} \t mean(l0) : {o_bestl0[success_mask].mean():.6f} \t Current Learning Rate: {current_lr:.3f}")

        if outer_step != (binary_search_steps - 1):
            #o_success_attack  = torch.logical_or(o_success_attack, success_mask)
            o_success_attack, o_l0dist = compare_round(imgs,o_bestattack, model,removal_array)
            print(f"outer_step {outer_step + 1}: all success : {o_success_attack.sum()} \t mean(l0)(success) : {o_l0dist[o_success_attack].mean():.4f}")
            #print('o_success_attack ', o_success_attack.sum())
            print('-------------------------------------------------------------')
            failure_mask = ~success_mask
            upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
            lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

            upper_limit_mask = upper_bound < 1e10
            CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
            CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    success_attack_mask = (o_bestl0 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output